# Second Application — Full benchmark (validation of Sena's update)

This notebook runs the same 15-algorithm benchmark that produced
**Tables 1–4** in `Sena_New_Update.pdf`, but on the **second
application's** seismic ln(Sa) maps instead of the lognormal random
process. The point is to check whether the numerical findings (e.g.
`RP-CE-LE` near-resolution-independence, `KN` ~+61.7% RD, MB constant
~+0.67% RD, LK overhead) reproduce on a different dataset.

**Setup, mirrored from the first-app benchmark:**

* `Nsim = 3000`, `N_QUANTA = 50`, `N_EXP = 100` seeds per (algo, R)
* `R_LIST = [128, 256, 512, 1024, 2048, 4096, 8192, 16384]` — the R sweep
  is done by **1-D interpolation along the flattened ln(Sa) maps**, the
  same way `interpolate_dataset(X, R)` does it in the first notebook.
  (R = 64000 is omitted by default — it's slow and not strictly needed
  for the validation; uncomment it in `R_LIST` if you want it.)
* `MAX_ITER = 20`, `tol = 1e-4` for capped iterative methods.

**Run order:** top to bottom. Each algorithm cell is independent of
every other algorithm cell — the only shared state is `X_log`,
`interpolate_dataset`, and the master settings cell.

> The benchmark on `R = 16384` for `HT` / `HK` is the slow part —
> roughly 30–60 min on a laptop with 100 seeds. If you want to ship a
> quicker sanity run, drop `16384` from `R_LIST` and re-run only those
> two cells.


In [ ]:
# STEP 1 — Generate seismic IM maps for the paper's second application
# Ready to paste into Jupyter / VS Code

import numpy as np
from dataclasses import dataclass

# ---------------------------
# 1. Basic geometry helpers
# ---------------------------

EARTH_RADIUS_KM = 6371.0

def latlon_to_xy_km(lat, lon, lat0, lon0):
    """
    Local equirectangular projection to x-y in km.
    lat, lon, lat0, lon0 in degrees.
    lon should be negative for west longitude.
    """
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    lat0 = float(lat0)
    lon0 = float(lon0)

    x = (np.radians(lon - lon0) *
         EARTH_RADIUS_KM *
         np.cos(np.radians((lat + lat0) / 2.0)))
    y = np.radians(lat - lat0) * EARTH_RADIUS_KM
    return x, y


def sample_triangular(left, mode, right, size=None, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    return rng.triangular(left, mode, right, size=size)


def sample_point_on_segment(p0, p1, rng=None):
    """
    Uniform sample along a line segment.
    p0, p1 are (x, y).
    """
    rng = np.random.default_rng() if rng is None else rng
    t = rng.uniform(0.0, 1.0)
    return (1 - t) * np.asarray(p0) + t * np.asarray(p1)


def segment_unit_direction(p0, p1):
    v = np.asarray(p1) - np.asarray(p0)
    nrm = np.linalg.norm(v)
    if nrm == 0:
        raise ValueError("Fault segment has zero length.")
    return v / nrm


def rupture_segment_from_epicenter(epicenter_xy, fault_p0, fault_p1, rupture_length_km):
    """
    Rupture centered on the sampled epicenter and extending symmetrically
    along the fault strike, per the paper text.
    """
    u = segment_unit_direction(fault_p0, fault_p1)
    half = 0.5 * rupture_length_km
    start = epicenter_xy - half * u
    end = epicenter_xy + half * u
    return start, end


def point_to_segment_distance(points_xy, seg_a, seg_b):
    """
    Distance from each point to a line segment in 2D.
    points_xy: (n, 2)
    seg_a, seg_b: (2,)
    returns: (n,)
    """
    p = np.asarray(points_xy, dtype=float)
    a = np.asarray(seg_a, dtype=float)
    b = np.asarray(seg_b, dtype=float)

    ab = b - a
    ab2 = np.dot(ab, ab)
    if ab2 == 0:
        return np.linalg.norm(p - a, axis=1)

    ap = p - a
    t = (ap @ ab) / ab2
    t = np.clip(t, 0.0, 1.0)
    proj = a + np.outer(t, ab)
    return np.linalg.norm(p - proj, axis=1)


# ---------------------------
# 2. Paper-specific parameters
# ---------------------------

# Fault coordinates from Table 6 in the paper
# Convert west longitudes to negative values
FAULT_LATLON = {
    "A": (34.29, -118.39),
    "B": (34.26, -118.37),
    "C": (34.24, -118.36),
    "D": (34.27, -118.31),
}

# Paper figure bounds (approx from Fig. 10–13)
LAT_MIN, LAT_MAX = 34.20, 34.34
LON_MIN, LON_MAX = -118.45, -118.275

# Abrahamson–Silva (1997) coefficients at T = 0.1 s from the paper
C4 = 5.50
A1 = 2.160
A2 = 0.512
A3 = -1.145
A12 = 0.028
A13 = 0.17
C1 = 6.4
N_EXP = 2

# Important note:
# The paper excerpt gives the residual correlation length range (~10–40 km)
# but does not pin down one single b_epsilon value in the visible text.
# So for a first debug version, choose a reasonable middle value.
B_EPS_KM = 20.0

# Same issue for residual standard deviation: not specified in the visible setup text.
# Use a placeholder that you can tune later.
SIGMA_EPS = 0.60

# Grid size for first debug pass
NX, NY = 20, 20   # 400 grid points total

# Reproduction-style MCS size from Fig. 12
NSIM_DEBUG = 3000


# ---------------------------
# 3. Build spatial grid
# ---------------------------

lon_vec = np.linspace(LON_MIN, LON_MAX, NX)
lat_vec = np.linspace(LAT_MIN, LAT_MAX, NY)
lon_grid, lat_grid = np.meshgrid(lon_vec, lat_vec)

lat0 = 0.5 * (LAT_MIN + LAT_MAX)
lon0 = 0.5 * (LON_MIN + LON_MAX)

x_grid, y_grid = latlon_to_xy_km(lat_grid, lon_grid, lat0, lon0)
points_xy = np.column_stack([x_grid.ravel(), y_grid.ravel()])

# Convert fault nodes to local x-y
fault_xy = {}
for k, (lat, lon) in FAULT_LATLON.items():
    xk, yk = latlon_to_xy_km(lat, lon, lat0, lon0)
    fault_xy[k] = np.array([xk, yk], dtype=float)

fault_segments = {
    "AB": (fault_xy["A"], fault_xy["B"]),
    "CD": (fault_xy["C"], fault_xy["D"]),
}


# ---------------------------
# 4. Ground-motion model
# ---------------------------

def rupture_length_km_from_magnitude(M):
    """
    HAZUS-MH regression from the paper:
    log10(L) = -3.55 + 0.74 M
    """
    return 10 ** (-3.55 + 0.74 * M)


def abrahamson_silva_ln_sa(M, rrup_km):
    """
    Paper Eq. (22) as written in the visible excerpt:
    ln Sa = a1 + a2(M-c1) + a12(8.5-M)^n + [a3 + a13(M-c1)] ln(Reff)
    with Reff = sqrt(rrup^2 + c4^2)
    """
    rrup_km = np.asarray(rrup_km, dtype=float)
    reff = np.sqrt(rrup_km**2 + C4**2)

    ln_sa = (
        A1
        + A2 * (M - C1)
        + A12 * (8.5 - M) ** N_EXP
        + (A3 + A13 * (M - C1)) * np.log(reff)
    )
    return ln_sa


# ---------------------------
# 5. Spatially correlated residual field
# ---------------------------

def exponential_covariance(points_xy, b_eps_km=B_EPS_KM, sigma_eps=SIGMA_EPS, jitter=1e-8):
    """
    Cov_ij = sigma^2 * exp(-d_ij / b_eps)
    """
    diff = points_xy[:, None, :] - points_xy[None, :, :]
    dist = np.sqrt(np.sum(diff**2, axis=2))
    cov = (sigma_eps**2) * np.exp(-dist / b_eps_km)
    cov.flat[:: cov.shape[0] + 1] += jitter
    return cov


COV_EPS = exponential_covariance(points_xy)
CHOLESKY_EPS = np.linalg.cholesky(COV_EPS)


def sample_correlated_epsilon(rng=None):
    """
    Gaussian field with exponential covariance.
    The paper mentions SRM for epsilon generation.
    This Cholesky version is easier for a first faithful debug build.
    """
    rng = np.random.default_rng() if rng is None else rng
    z = rng.standard_normal(points_xy.shape[0])
    eps = CHOLESKY_EPS @ z
    return eps


# ---------------------------
# 6. One seismic IM map realization
# ---------------------------

@dataclass
class EventSample:
    fault_name: str
    epicenter_xy: np.ndarray
    magnitude: float
    depth_km: float
    rupture_length_km: float
    rupture_start_xy: np.ndarray
    rupture_end_xy: np.ndarray


def sample_event(rng=None):
    rng = np.random.default_rng() if rng is None else rng

    fault_name = "AB" if rng.uniform() < 0.5 else "CD"
    f0, f1 = fault_segments[fault_name]

    epicenter_xy = sample_point_on_segment(f0, f1, rng=rng)
    M = float(sample_triangular(5.5, 6.0, 6.5, rng=rng))
    depth_km = float(sample_triangular(2.0, 4.0, 6.0, rng=rng))
    L_km = float(rupture_length_km_from_magnitude(M))
    r0, r1 = rupture_segment_from_epicenter(epicenter_xy, f0, f1, L_km)

    return EventSample(
        fault_name=fault_name,
        epicenter_xy=epicenter_xy,
        magnitude=M,
        depth_km=depth_km,
        rupture_length_km=L_km,
        rupture_start_xy=r0,
        rupture_end_xy=r1,
    )


def simulate_one_ln_sa_map(rng=None):
    """
    Returns ln(Sa) on the grid, flattened to shape (R,).
    """
    rng = np.random.default_rng() if rng is None else rng
    ev = sample_event(rng=rng)

    horizontal_dist = point_to_segment_distance(
        points_xy,
        ev.rupture_start_xy,
        ev.rupture_end_xy
    )

    # Simplified rrup for vertical strike-slip rupture plane:
    rrup = np.sqrt(horizontal_dist**2 + ev.depth_km**2)

    ln_sa_median = abrahamson_silva_ln_sa(ev.magnitude, rrup)
    eps = sample_correlated_epsilon(rng=rng)

    ln_sa = ln_sa_median + eps
    return ln_sa, ev


def simulate_many_maps(nsim=NSIM_DEBUG, rng_seed=42):
    """
    Generates:
      X_log  : (nsim, R) matrix of ln(Sa) maps
      X_sa   : (nsim, R) matrix of Sa maps
      events : list of sampled event metadata
    """
    rng = np.random.default_rng(rng_seed)

    X_log = np.zeros((nsim, points_xy.shape[0]), dtype=float)
    events = []

    for i in range(nsim):
        ln_sa_i, ev = simulate_one_ln_sa_map(rng=rng)
        X_log[i, :] = ln_sa_i
        events.append(ev)

        if (i + 1) % 200 == 0:
            print(f"Generated {i+1}/{nsim} maps")

    X_sa = np.exp(X_log)
    return X_log, X_sa, events


# ---------------------------
# 7. Run Step 1
# ---------------------------

X_log, X_sa, events = simulate_many_maps(nsim=NSIM_DEBUG, rng_seed=123)

print("\nDone.")
print("X_log shape:", X_log.shape)   # should be (2000, 400) for debug version
print("X_sa  shape:", X_sa.shape)
print("Grid points R:", points_xy.shape[0])
print("Sample map min/max Sa:", float(X_sa[0].min()), float(X_sa[0].max()))
print("First event:", events[0])


## Interpolation along R

`interpolate_dataset(X, R_new)` lifts each ln(Sa) sample to resolution `R_new` by 1-D linear interpolation along its flattened axis. This is exactly the function used in the first-application notebook, so the R sweep is comparable.


In [ ]:
import numpy as np
import os
import gc

# Optional process memory tracking
try:
    import psutil
    PROCESS = psutil.Process(os.getpid())
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False


def format_bytes(nbytes):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if nbytes < 1024:
            return f"{nbytes:.2f} {unit}"
        nbytes /= 1024
    return f"{nbytes:.2f} PB"


def print_process_memory():
    if HAS_PSUTIL:
        rss = PROCESS.memory_info().rss
        print("Current process RAM:", format_bytes(rss))
    else:
        print("Current process RAM: psutil not installed")


def interpolate_dataset(X, R_new):
    """
    Identical contract to the first-app notebook: lift each row of X
    from its current resolution to R_new via linear interpolation.
    """
    X = np.asarray(X, dtype=float)
    N, R_old = X.shape
    if R_new == R_old:
        return X.copy()
    old_grid = np.linspace(0, 1, R_old)
    new_grid = np.linspace(0, 1, R_new)
    X_new = np.empty((N, R_new), dtype=float)
    for i in range(N):
        X_new[i] = np.interp(new_grid, old_grid, X[i])
    return X_new


# We benchmark on the LOG-INTENSITY maps, exactly as in the first app.
# (Sena's first-app benchmark also clusters in log-space because the
# field is lognormal.)
X = X_log

print("Source X shape:", X.shape)
print("R_original:", X.shape[1])

# Quick interpolation test
for R_test in [128, 1024, 8192]:
    Xr = interpolate_dataset(X, R_test)
    print(f"  interpolated to R={R_test}: shape={Xr.shape}, "
          f"min={Xr.min():.3f}, max={Xr.max():.3f}, "
          f"memory={format_bytes(Xr.nbytes)}")
    del Xr
    gc.collect()


## Master settings

Mirrors the first-application notebook so the validation is apples-to-apples.


In [ ]:
import numpy as np

# Match Sena's first-app benchmark
R_LIST   = [128, 256, 512, 1024, 2048, 4096, 8192, 16384]
N_EXP    = 100         # 100 seeds per (algorithm, R)
N_QUANTA = 50          # paper's N
MAX_ITER = 20          # niter <= 20 budget
TOL      = 1e-4

print("Master settings loaded:")
print("  R_LIST   =", R_LIST)
print("  N_EXP    =", N_EXP)
print("  N_QUANTA =", N_QUANTA)
print("  MAX_ITER =", MAX_ITER)
print("  TOL      =", TOL)
print("  X shape  =", X.shape)


## LX — Lloyd classic (random init)

Baseline. Random initial quanta + Lloyd iterations, capped at 20 iters.


In [ ]:
import time, gc, numpy as np
from sklearn.cluster import KMeans

LX_all_times = []
LX_all_sse   = []

print("Collecting LX per-experiment times (seismic data)...")
total_start = time.perf_counter()

for iR, R in enumerate(R_LIST, 1):
    print(f"\n========== LX | R={R} ({iR}/{len(R_LIST)}) ==========")
    Xr = interpolate_dataset(X, R)
    print(f"  Xr shape: {Xr.shape} | mem: {format_bytes(Xr.nbytes)}")

    exp_times, exp_sse = [], []
    R_start = time.perf_counter()

    for exp in range(N_EXP):
        t0 = time.perf_counter()
        km = KMeans(
            n_clusters=N_QUANTA, init="random", n_init=1,
            max_iter=MAX_ITER, tol=TOL, algorithm="lloyd",
            random_state=exp,
        )
        km.fit(Xr)
        dt = time.perf_counter() - t0
        exp_times.append(dt)
        exp_sse.append(float(km.inertia_))

        if (exp + 1) % 10 == 0 or exp == N_EXP - 1:
            avg = float(np.mean(exp_times))
            eta = (N_EXP - exp - 1) * avg
            print(f"  [R={R}] EXP {exp+1:3d}/{N_EXP} | t={dt:.4f}s | "
                  f"avg={avg:.4f}s | ETA={eta:.1f}s | iter={km.n_iter_} | "
                  f"SSE={km.inertia_:.3e}")

        del km

    LX_all_times.append(exp_times)
    LX_all_sse.append(exp_sse)

    m, s = float(np.mean(exp_times)), float(np.std(exp_times, ddof=1))
    cv   = s / m if m > 0 else float("nan")
    print(f"  R={R} DONE | mean={m:.4f}s | std={s:.4f}s | CV={cv:.4f} | "
          f"R-time={time.perf_counter()-R_start:.1f}s")

    del Xr; gc.collect()

print(f"\n========== LX COMPLETE ({(time.perf_counter()-total_start)/60:.2f} min) ==========")


In [ ]:
# ============================================================
# BOX PLOTS for LX (second application, seismic data)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

assert "LX_all_times" in globals(), "LX_all_times not found. Run the LX block first."

# --- Box plot: time per R ---
time_data = [list(np.asarray(t, dtype=float)) for t in LX_all_times]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(
    time_data,
    showmeans=True, meanline=True, patch_artist=True,
    boxprops=dict(facecolor="#378ADD", alpha=0.4, edgecolor="#378ADD"),
    medianprops=dict(color="black", linewidth=1.5),
    meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
    flierprops=dict(marker="o", markersize=3, markerfacecolor="#378ADD", alpha=0.5),
)
ax.set_xticklabels([str(int(R)) for R in R_LIST])
ax.set_xlabel("Discretization (R)")
ax.set_ylabel("Time per experiment (sec)")
ax.set_title(f"LX (seismic data) — time distribution across {N_EXP} seeds per R")
ax.set_yscale("log")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("lx_boxplot_time.png", dpi=150)
plt.show()

# --- Box plot: SSE per R ---
sse_data_clean = []
for s in LX_all_sse:
    arr = np.asarray(s, dtype=float)
    arr = arr[~np.isnan(arr)]
    sse_data_clean.append(arr.tolist())

if any(len(s) > 0 for s in sse_data_clean):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(
        sse_data_clean,
        showmeans=True, meanline=True, patch_artist=True,
        boxprops=dict(facecolor="#378ADD", alpha=0.4, edgecolor="#378ADD"),
        medianprops=dict(color="black", linewidth=1.5),
        meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
        flierprops=dict(marker="o", markersize=3, markerfacecolor="#378ADD", alpha=0.5),
    )
    ax.set_xticklabels([str(int(R)) for R in R_LIST])
    ax.set_xlabel("Discretization (R)")
    ax.set_ylabel("SSE per experiment")
    ax.set_title(f"LX (seismic data) — SSE distribution across {N_EXP} seeds per R")
    ax.set_yscale("log")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("lx_boxplot_sse.png", dpi=150)
    plt.show()

print("LX box plots saved.")


## LE — Lloyd with Elkan's triangle inequality

Same iteration cap as LX; uses `algorithm="elkan"`.


In [ ]:
import time, gc, numpy as np
from sklearn.cluster import KMeans

LE_all_times = []
LE_all_sse   = []

print("Collecting LE per-experiment times...")
total_start = time.perf_counter()

for iR, R in enumerate(R_LIST, 1):
    print(f"\n========== LE | R={R} ({iR}/{len(R_LIST)}) ==========")
    Xr = interpolate_dataset(X, R)
    print(f"  Xr shape: {Xr.shape} | mem: {format_bytes(Xr.nbytes)}")

    exp_times, exp_sse = [], []
    for exp in range(N_EXP):
        t0 = time.perf_counter()
        km = KMeans(
            n_clusters=N_QUANTA, init="k-means++", n_init=1,
            max_iter=MAX_ITER, tol=TOL, algorithm="elkan",
            random_state=exp,
        )
        km.fit(Xr)
        dt = time.perf_counter() - t0
        exp_times.append(dt)
        exp_sse.append(float(km.inertia_))
        if (exp + 1) % 10 == 0 or exp == N_EXP - 1:
            avg = float(np.mean(exp_times))
            print(f"  [R={R}] EXP {exp+1:3d}/{N_EXP} | t={dt:.4f}s | "
                  f"avg={avg:.4f}s | iter={km.n_iter_} | SSE={km.inertia_:.3e}")
        del km

    LE_all_times.append(exp_times)
    LE_all_sse.append(exp_sse)

    m = float(np.mean(exp_times)); s = float(np.std(exp_times, ddof=1))
    print(f"  R={R} DONE | mean={m:.4f}s | std={s:.4f}s | CV={s/m:.4f}")

    del Xr; gc.collect()

print(f"\n========== LE COMPLETE ({(time.perf_counter()-total_start)/60:.2f} min) ==========")


In [ ]:
# ============================================================
# BOX PLOTS for LE (second application, seismic data)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

assert "LE_all_times" in globals(), "LE_all_times not found. Run the LE block first."

# --- Box plot: time per R ---
time_data = [list(np.asarray(t, dtype=float)) for t in LE_all_times]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(
    time_data,
    showmeans=True, meanline=True, patch_artist=True,
    boxprops=dict(facecolor="#1D9E75", alpha=0.4, edgecolor="#1D9E75"),
    medianprops=dict(color="black", linewidth=1.5),
    meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
    flierprops=dict(marker="o", markersize=3, markerfacecolor="#1D9E75", alpha=0.5),
)
ax.set_xticklabels([str(int(R)) for R in R_LIST])
ax.set_xlabel("Discretization (R)")
ax.set_ylabel("Time per experiment (sec)")
ax.set_title(f"LE (seismic data) — time distribution across {N_EXP} seeds per R")
ax.set_yscale("log")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("le_boxplot_time.png", dpi=150)
plt.show()

# --- Box plot: SSE per R ---
sse_data_clean = []
for s in LE_all_sse:
    arr = np.asarray(s, dtype=float)
    arr = arr[~np.isnan(arr)]
    sse_data_clean.append(arr.tolist())

if any(len(s) > 0 for s in sse_data_clean):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(
        sse_data_clean,
        showmeans=True, meanline=True, patch_artist=True,
        boxprops=dict(facecolor="#1D9E75", alpha=0.4, edgecolor="#1D9E75"),
        medianprops=dict(color="black", linewidth=1.5),
        meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
        flierprops=dict(marker="o", markersize=3, markerfacecolor="#1D9E75", alpha=0.5),
    )
    ax.set_xticklabels([str(int(R)) for R in R_LIST])
    ax.set_xlabel("Discretization (R)")
    ax.set_ylabel("SSE per experiment")
    ax.set_title(f"LE (seismic data) — SSE distribution across {N_EXP} seeds per R")
    ax.set_yscale("log")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("le_boxplot_sse.png", dpi=150)
    plt.show()

print("LE box plots saved.")


## LK — KN initialization + Lloyd refinement (Sena's hybrid)

Builds initial quanta via the KN subset-init logic and then runs Lloyd. Tracks `n_iter_` so we can compare with Sena's Finding 5 (LK overhead vs iteration reduction).


In [ ]:
import time, gc, numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min

LK_all_times = []
LK_all_sse   = []
LK_all_iters = []

def kn_init_centers(Xr, k, seed):
    rng = np.random.default_rng(seed)
    n_sub = min(Xr.shape[0], 4 * k)
    sub_idx = rng.choice(Xr.shape[0], size=n_sub, replace=False)
    subset = Xr[sub_idx]
    init_idx = rng.choice(subset.shape[0], size=k, replace=False)
    return subset[init_idx].copy()

print("Collecting LK per-experiment times...")
total_start = time.perf_counter()

for iR, R in enumerate(R_LIST, 1):
    print(f"\n========== LK | R={R} ({iR}/{len(R_LIST)}) ==========")
    Xr = interpolate_dataset(X, R)
    print(f"  Xr shape: {Xr.shape} | mem: {format_bytes(Xr.nbytes)}")

    exp_times, exp_sse, exp_iters = [], [], []
    for exp in range(N_EXP):
        t0 = time.perf_counter()
        # KN initialization phase
        C0 = kn_init_centers(Xr, N_QUANTA, seed=exp)
        # Lloyd refinement
        km = KMeans(
            n_clusters=N_QUANTA, init=C0, n_init=1,
            max_iter=MAX_ITER, tol=TOL, algorithm="lloyd",
            random_state=exp,
        )
        km.fit(Xr)
        dt = time.perf_counter() - t0
        exp_times.append(dt)
        exp_sse.append(float(km.inertia_))
        exp_iters.append(int(km.n_iter_))
        if (exp + 1) % 10 == 0 or exp == N_EXP - 1:
            avg = float(np.mean(exp_times))
            print(f"  [R={R}] EXP {exp+1:3d}/{N_EXP} | t={dt:.4f}s | "
                  f"avg={avg:.4f}s | iter={km.n_iter_} | SSE={km.inertia_:.3e}")
        del km

    LK_all_times.append(exp_times)
    LK_all_sse.append(exp_sse)
    LK_all_iters.append(exp_iters)

    m = float(np.mean(exp_times)); s = float(np.std(exp_times, ddof=1))
    mi = float(np.mean(exp_iters))
    print(f"  R={R} DONE | mean={m:.4f}s | std={s:.4f}s | CV={s/m:.4f} | "
          f"mean_iter={mi:.2f}")

    del Xr; gc.collect()

print(f"\n========== LK COMPLETE ({(time.perf_counter()-total_start)/60:.2f} min) ==========")


In [ ]:
# ============================================================
# BOX PLOTS for LK (second application, seismic data)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

assert "LK_all_times" in globals(), "LK_all_times not found. Run the LK block first."

# --- Box plot: time per R ---
time_data = [list(np.asarray(t, dtype=float)) for t in LK_all_times]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(
    time_data,
    showmeans=True, meanline=True, patch_artist=True,
    boxprops=dict(facecolor="#BA7517", alpha=0.4, edgecolor="#BA7517"),
    medianprops=dict(color="black", linewidth=1.5),
    meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
    flierprops=dict(marker="o", markersize=3, markerfacecolor="#BA7517", alpha=0.5),
)
ax.set_xticklabels([str(int(R)) for R in R_LIST])
ax.set_xlabel("Discretization (R)")
ax.set_ylabel("Time per experiment (sec)")
ax.set_title(f"LK (seismic data) — time distribution across {N_EXP} seeds per R")
ax.set_yscale("log")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("lk_boxplot_time.png", dpi=150)
plt.show()

# --- Box plot: SSE per R ---
sse_data_clean = []
for s in LK_all_sse:
    arr = np.asarray(s, dtype=float)
    arr = arr[~np.isnan(arr)]
    sse_data_clean.append(arr.tolist())

if any(len(s) > 0 for s in sse_data_clean):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(
        sse_data_clean,
        showmeans=True, meanline=True, patch_artist=True,
        boxprops=dict(facecolor="#BA7517", alpha=0.4, edgecolor="#BA7517"),
        medianprops=dict(color="black", linewidth=1.5),
        meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
        flierprops=dict(marker="o", markersize=3, markerfacecolor="#BA7517", alpha=0.5),
    )
    ax.set_xticklabels([str(int(R)) for R in R_LIST])
    ax.set_xlabel("Discretization (R)")
    ax.set_ylabel("SSE per experiment")
    ax.set_title(f"LK (seismic data) — SSE distribution across {N_EXP} seeds per R")
    ax.set_yscale("log")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("lk_boxplot_sse.png", dpi=150)
    plt.show()

print("LK box plots saved.")


## KN — One-pass nearest-neighbour assignment

No iterative refinement. Random subset → centers → single-pass distance assignment → SSE.


In [ ]:
import time, gc, numpy as np

KN_all_times = []
KN_all_sse   = []

def kn_onepass_sse(Xr, K, seed):
    rng = np.random.default_rng(seed)
    n = Xr.shape[0]
    idx = rng.choice(n, size=K, replace=False)
    C = Xr[idx]
    x2 = np.sum(Xr * Xr, axis=1, keepdims=True)
    c2 = np.sum(C * C, axis=1, keepdims=True).T
    D2 = x2 + c2 - 2.0 * (Xr @ C.T)
    D2 = np.maximum(D2, 0.0)
    return float(np.sum(np.min(D2, axis=1)))

print("Collecting KN per-experiment times...")
total_start = time.perf_counter()

for iR, R in enumerate(R_LIST, 1):
    print(f"\n========== KN | R={R} ({iR}/{len(R_LIST)}) ==========")
    Xr = interpolate_dataset(X, R)
    exp_times, exp_sse = [], []
    for exp in range(N_EXP):
        t0 = time.perf_counter()
        sse = kn_onepass_sse(Xr, N_QUANTA, seed=exp)
        dt = time.perf_counter() - t0
        exp_times.append(dt); exp_sse.append(sse)
        if (exp + 1) % 20 == 0 or exp == N_EXP - 1:
            print(f"  [R={R}] EXP {exp+1:3d}/{N_EXP} | t={dt:.5f}s | SSE={sse:.3e}")
    KN_all_times.append(exp_times)
    KN_all_sse.append(exp_sse)
    m = float(np.mean(exp_times)); s = float(np.std(exp_times, ddof=1))
    print(f"  R={R} DONE | mean={m:.5f}s | std={s:.5f}s | CV={s/m:.4f}")
    del Xr; gc.collect()

print(f"\n========== KN COMPLETE ({(time.perf_counter()-total_start)/60:.2f} min) ==========")


In [ ]:
# ============================================================
# BOX PLOTS for KN (second application, seismic data)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

assert "KN_all_times" in globals(), "KN_all_times not found. Run the KN block first."

# --- Box plot: time per R ---
time_data = [list(np.asarray(t, dtype=float)) for t in KN_all_times]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(
    time_data,
    showmeans=True, meanline=True, patch_artist=True,
    boxprops=dict(facecolor="#D85A30", alpha=0.4, edgecolor="#D85A30"),
    medianprops=dict(color="black", linewidth=1.5),
    meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
    flierprops=dict(marker="o", markersize=3, markerfacecolor="#D85A30", alpha=0.5),
)
ax.set_xticklabels([str(int(R)) for R in R_LIST])
ax.set_xlabel("Discretization (R)")
ax.set_ylabel("Time per experiment (sec)")
ax.set_title(f"KN (seismic data) — time distribution across {N_EXP} seeds per R")
ax.set_yscale("log")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("kn_boxplot_time.png", dpi=150)
plt.show()

# --- Box plot: SSE per R ---
sse_data_clean = []
for s in KN_all_sse:
    arr = np.asarray(s, dtype=float)
    arr = arr[~np.isnan(arr)]
    sse_data_clean.append(arr.tolist())

if any(len(s) > 0 for s in sse_data_clean):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(
        sse_data_clean,
        showmeans=True, meanline=True, patch_artist=True,
        boxprops=dict(facecolor="#D85A30", alpha=0.4, edgecolor="#D85A30"),
        medianprops=dict(color="black", linewidth=1.5),
        meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
        flierprops=dict(marker="o", markersize=3, markerfacecolor="#D85A30", alpha=0.5),
    )
    ax.set_xticklabels([str(int(R)) for R in R_LIST])
    ax.set_xlabel("Discretization (R)")
    ax.set_ylabel("SSE per experiment")
    ax.set_title(f"KN (seismic data) — SSE distribution across {N_EXP} seeds per R")
    ax.set_yscale("log")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("kn_boxplot_sse.png", dpi=150)
    plt.show()

print("KN box plots saved.")


## HT — Hierarchical (Ward agglomerative)

Note: HT does not produce inertia natively; SSE is computed in-place from the cluster labels for fair comparison with the other methods.


In [ ]:
import time, gc, numpy as np
from sklearn.cluster import AgglomerativeClustering

HT_all_times = []
HT_all_sse   = []

def labels_to_sse(Xr, labels, K):
    sse = 0.0
    for k in range(K):
        idx = np.where(labels == k)[0]
        if idx.size == 0:
            continue
        center = Xr[idx].mean(axis=0)
        diffs = Xr[idx] - center
        sse += float(np.sum(diffs * diffs))
    return sse

print("Collecting HT per-experiment times...")
print("WARNING: HT scales like O((Nsim - N)^2 * R). At R = 16384 a single seed")
print("         takes a while; expect this cell to be the slowest by far.")
total_start = time.perf_counter()

for iR, R in enumerate(R_LIST, 1):
    print(f"\n========== HT | R={R} ({iR}/{len(R_LIST)}) ==========")
    Xr = interpolate_dataset(X, R)
    print(f"  Xr shape: {Xr.shape} | mem: {format_bytes(Xr.nbytes)}")

    exp_times, exp_sse = [], []
    for exp in range(N_EXP):
        t0 = time.perf_counter()
        ag = AgglomerativeClustering(n_clusters=N_QUANTA, linkage="ward")
        labels = ag.fit_predict(Xr)
        sse = labels_to_sse(Xr, labels, N_QUANTA)
        dt = time.perf_counter() - t0
        exp_times.append(dt); exp_sse.append(sse)
        if (exp + 1) % 5 == 0 or exp == N_EXP - 1:
            avg = float(np.mean(exp_times))
            eta = (N_EXP - exp - 1) * avg
            print(f"  [R={R}] EXP {exp+1:3d}/{N_EXP} | t={dt:.4f}s | "
                  f"avg={avg:.4f}s | ETA={eta:.1f}s | SSE={sse:.3e}")
        del ag, labels

    HT_all_times.append(exp_times)
    HT_all_sse.append(exp_sse)
    m = float(np.mean(exp_times)); s = float(np.std(exp_times, ddof=1))
    print(f"  R={R} DONE | mean={m:.4f}s | std={s:.4f}s | CV={s/m:.4f}")
    del Xr; gc.collect()

print(f"\n========== HT COMPLETE ({(time.perf_counter()-total_start)/60:.2f} min) ==========")


In [ ]:
# ============================================================
# BOX PLOTS for HT (second application, seismic data)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

assert "HT_all_times" in globals(), "HT_all_times not found. Run the HT block first."

# --- Box plot: time per R ---
time_data = [list(np.asarray(t, dtype=float)) for t in HT_all_times]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(
    time_data,
    showmeans=True, meanline=True, patch_artist=True,
    boxprops=dict(facecolor="#7F77DD", alpha=0.4, edgecolor="#7F77DD"),
    medianprops=dict(color="black", linewidth=1.5),
    meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
    flierprops=dict(marker="o", markersize=3, markerfacecolor="#7F77DD", alpha=0.5),
)
ax.set_xticklabels([str(int(R)) for R in R_LIST])
ax.set_xlabel("Discretization (R)")
ax.set_ylabel("Time per experiment (sec)")
ax.set_title(f"HT (seismic data) — time distribution across {N_EXP} seeds per R")
ax.set_yscale("log")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("ht_boxplot_time.png", dpi=150)
plt.show()

# --- Box plot: SSE per R ---
sse_data_clean = []
for s in HT_all_sse:
    arr = np.asarray(s, dtype=float)
    arr = arr[~np.isnan(arr)]
    sse_data_clean.append(arr.tolist())

if any(len(s) > 0 for s in sse_data_clean):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(
        sse_data_clean,
        showmeans=True, meanline=True, patch_artist=True,
        boxprops=dict(facecolor="#7F77DD", alpha=0.4, edgecolor="#7F77DD"),
        medianprops=dict(color="black", linewidth=1.5),
        meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
        flierprops=dict(marker="o", markersize=3, markerfacecolor="#7F77DD", alpha=0.5),
    )
    ax.set_xticklabels([str(int(R)) for R in R_LIST])
    ax.set_xlabel("Discretization (R)")
    ax.set_ylabel("SSE per experiment")
    ax.set_title(f"HT (seismic data) — SSE distribution across {N_EXP} seeds per R")
    ax.set_yscale("log")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("ht_boxplot_sse.png", dpi=150)
    plt.show()

print("HT box plots saved.")


## HK — Hierarchical-init + Lloyd refinement


In [ ]:
import time, gc, numpy as np
from sklearn.cluster import AgglomerativeClustering, KMeans

HK_all_times = []
HK_all_sse   = []

print("Collecting HK per-experiment times...")
total_start = time.perf_counter()

for iR, R in enumerate(R_LIST, 1):
    print(f"\n========== HK | R={R} ({iR}/{len(R_LIST)}) ==========")
    Xr = interpolate_dataset(X, R)
    exp_times, exp_sse = [], []
    for exp in range(N_EXP):
        t0 = time.perf_counter()
        ag = AgglomerativeClustering(n_clusters=N_QUANTA, linkage="ward")
        labels = ag.fit_predict(Xr)
        centroids = np.array([
            Xr[labels == k].mean(axis=0) if (labels == k).any() else Xr[0]
            for k in range(N_QUANTA)
        ])
        km = KMeans(
            n_clusters=N_QUANTA, init=centroids, n_init=1,
            max_iter=MAX_ITER, tol=TOL, algorithm="lloyd",
            random_state=exp,
        )
        km.fit(Xr)
        dt = time.perf_counter() - t0
        exp_times.append(dt); exp_sse.append(float(km.inertia_))
        if (exp + 1) % 5 == 0 or exp == N_EXP - 1:
            avg = float(np.mean(exp_times))
            eta = (N_EXP - exp - 1) * avg
            print(f"  [R={R}] EXP {exp+1:3d}/{N_EXP} | t={dt:.4f}s | "
                  f"avg={avg:.4f}s | ETA={eta:.1f}s | iter={km.n_iter_} | "
                  f"SSE={km.inertia_:.3e}")
        del ag, labels, centroids, km

    HK_all_times.append(exp_times)
    HK_all_sse.append(exp_sse)
    m = float(np.mean(exp_times)); s = float(np.std(exp_times, ddof=1))
    print(f"  R={R} DONE | mean={m:.4f}s | std={s:.4f}s | CV={s/m:.4f}")
    del Xr; gc.collect()

print(f"\n========== HK COMPLETE ({(time.perf_counter()-total_start)/60:.2f} min) ==========")


In [ ]:
# ============================================================
# BOX PLOTS for HK (second application, seismic data)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

assert "HK_all_times" in globals(), "HK_all_times not found. Run the HK block first."

# --- Box plot: time per R ---
time_data = [list(np.asarray(t, dtype=float)) for t in HK_all_times]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(
    time_data,
    showmeans=True, meanline=True, patch_artist=True,
    boxprops=dict(facecolor="#D4537E", alpha=0.4, edgecolor="#D4537E"),
    medianprops=dict(color="black", linewidth=1.5),
    meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
    flierprops=dict(marker="o", markersize=3, markerfacecolor="#D4537E", alpha=0.5),
)
ax.set_xticklabels([str(int(R)) for R in R_LIST])
ax.set_xlabel("Discretization (R)")
ax.set_ylabel("Time per experiment (sec)")
ax.set_title(f"HK (seismic data) — time distribution across {N_EXP} seeds per R")
ax.set_yscale("log")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("hk_boxplot_time.png", dpi=150)
plt.show()

# --- Box plot: SSE per R ---
sse_data_clean = []
for s in HK_all_sse:
    arr = np.asarray(s, dtype=float)
    arr = arr[~np.isnan(arr)]
    sse_data_clean.append(arr.tolist())

if any(len(s) > 0 for s in sse_data_clean):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(
        sse_data_clean,
        showmeans=True, meanline=True, patch_artist=True,
        boxprops=dict(facecolor="#D4537E", alpha=0.4, edgecolor="#D4537E"),
        medianprops=dict(color="black", linewidth=1.5),
        meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
        flierprops=dict(marker="o", markersize=3, markerfacecolor="#D4537E", alpha=0.5),
    )
    ax.set_xticklabels([str(int(R)) for R in R_LIST])
    ax.set_xlabel("Discretization (R)")
    ax.set_ylabel("SSE per experiment")
    ax.set_title(f"HK (seismic data) — SSE distribution across {N_EXP} seeds per R")
    ax.set_yscale("log")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("hk_boxplot_sse.png", dpi=150)
    plt.show()

print("HK box plots saved.")


## MB — MiniBatchKMeans


In [ ]:
import time, gc, numpy as np
from sklearn.cluster import MiniBatchKMeans

MB_all_times = []
MB_all_sse   = []
BATCH_SIZE   = 1024

print("Collecting MB per-experiment times...")
total_start = time.perf_counter()

for iR, R in enumerate(R_LIST, 1):
    print(f"\n========== MB | R={R} ({iR}/{len(R_LIST)}) ==========")
    Xr = interpolate_dataset(X, R)
    exp_times, exp_sse = [], []
    for exp in range(N_EXP):
        t0 = time.perf_counter()
        mb = MiniBatchKMeans(
            n_clusters=N_QUANTA, init="k-means++", n_init=1,
            max_iter=MAX_ITER, tol=TOL, batch_size=BATCH_SIZE,
            random_state=exp,
        )
        mb.fit(Xr)
        dt = time.perf_counter() - t0
        exp_times.append(dt); exp_sse.append(float(mb.inertia_))
        if (exp + 1) % 10 == 0 or exp == N_EXP - 1:
            avg = float(np.mean(exp_times))
            print(f"  [R={R}] EXP {exp+1:3d}/{N_EXP} | t={dt:.4f}s | "
                  f"avg={avg:.4f}s | iter={mb.n_iter_} | SSE={mb.inertia_:.3e}")
        del mb

    MB_all_times.append(exp_times)
    MB_all_sse.append(exp_sse)
    m = float(np.mean(exp_times)); s = float(np.std(exp_times, ddof=1))
    print(f"  R={R} DONE | mean={m:.4f}s | std={s:.4f}s | CV={s/m:.4f}")
    del Xr; gc.collect()

print(f"\n========== MB COMPLETE ({(time.perf_counter()-total_start)/60:.2f} min) ==========")


In [ ]:
# ============================================================
# BOX PLOTS for MB (second application, seismic data)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

assert "MB_all_times" in globals(), "MB_all_times not found. Run the MB block first."

# --- Box plot: time per R ---
time_data = [list(np.asarray(t, dtype=float)) for t in MB_all_times]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(
    time_data,
    showmeans=True, meanline=True, patch_artist=True,
    boxprops=dict(facecolor="#639922", alpha=0.4, edgecolor="#639922"),
    medianprops=dict(color="black", linewidth=1.5),
    meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
    flierprops=dict(marker="o", markersize=3, markerfacecolor="#639922", alpha=0.5),
)
ax.set_xticklabels([str(int(R)) for R in R_LIST])
ax.set_xlabel("Discretization (R)")
ax.set_ylabel("Time per experiment (sec)")
ax.set_title(f"MB (seismic data) — time distribution across {N_EXP} seeds per R")
ax.set_yscale("log")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("mb_boxplot_time.png", dpi=150)
plt.show()

# --- Box plot: SSE per R ---
sse_data_clean = []
for s in MB_all_sse:
    arr = np.asarray(s, dtype=float)
    arr = arr[~np.isnan(arr)]
    sse_data_clean.append(arr.tolist())

if any(len(s) > 0 for s in sse_data_clean):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(
        sse_data_clean,
        showmeans=True, meanline=True, patch_artist=True,
        boxprops=dict(facecolor="#639922", alpha=0.4, edgecolor="#639922"),
        medianprops=dict(color="black", linewidth=1.5),
        meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
        flierprops=dict(marker="o", markersize=3, markerfacecolor="#639922", alpha=0.5),
    )
    ax.set_xticklabels([str(int(R)) for R in R_LIST])
    ax.set_xlabel("Discretization (R)")
    ax.set_ylabel("SSE per experiment")
    ax.set_title(f"MB (seismic data) — SSE distribution across {N_EXP} seeds per R")
    ax.set_yscale("log")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("mb_boxplot_sse.png", dpi=150)
    plt.show()

print("MB box plots saved.")


## ANN-FQ — Single-pass nearest-neighbour FQ

Same operational behaviour as KN (subset → 1-NN assign → SSE) but using `sklearn.neighbors.NearestNeighbors`. Sena's Finding 7 expects ANN-FQ ≈ KN at small R.


In [ ]:
import time, gc, numpy as np
from sklearn.neighbors import NearestNeighbors

ANN_all_times = []
ANN_all_sse   = []

print("Collecting ANN-FQ per-experiment times...")
total_start = time.perf_counter()

for iR, R in enumerate(R_LIST, 1):
    print(f"\n========== ANN-FQ | R={R} ({iR}/{len(R_LIST)}) ==========")
    Xr = interpolate_dataset(X, R)
    exp_times, exp_sse = [], []
    for exp in range(N_EXP):
        t0 = time.perf_counter()
        rng = np.random.default_rng(exp)
        idx = rng.choice(Xr.shape[0], size=N_QUANTA, replace=False)
        centers = Xr[idx]
        nn = NearestNeighbors(n_neighbors=1, algorithm="auto").fit(centers)
        d, _ = nn.kneighbors(Xr, return_distance=True)
        sse = float(np.sum(d.ravel() ** 2))
        dt = time.perf_counter() - t0
        exp_times.append(dt); exp_sse.append(sse)
        if (exp + 1) % 20 == 0 or exp == N_EXP - 1:
            print(f"  [R={R}] EXP {exp+1:3d}/{N_EXP} | t={dt:.5f}s | SSE={sse:.3e}")
        del centers, nn, d

    ANN_all_times.append(exp_times)
    ANN_all_sse.append(exp_sse)
    m = float(np.mean(exp_times)); s = float(np.std(exp_times, ddof=1))
    print(f"  R={R} DONE | mean={m:.5f}s | std={s:.5f}s | CV={s/m:.4f}")
    del Xr; gc.collect()

print(f"\n========== ANN-FQ COMPLETE ({(time.perf_counter()-total_start)/60:.2f} min) ==========")


In [ ]:
# ============================================================
# BOX PLOTS for ANN-FQ (second application, seismic data)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

assert "ANN_all_times" in globals(), "ANN_all_times not found. Run the ANN-FQ block first."

# --- Box plot: time per R ---
time_data = [list(np.asarray(t, dtype=float)) for t in ANN_all_times]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(
    time_data,
    showmeans=True, meanline=True, patch_artist=True,
    boxprops=dict(facecolor="#185FA5", alpha=0.4, edgecolor="#185FA5"),
    medianprops=dict(color="black", linewidth=1.5),
    meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
    flierprops=dict(marker="o", markersize=3, markerfacecolor="#185FA5", alpha=0.5),
)
ax.set_xticklabels([str(int(R)) for R in R_LIST])
ax.set_xlabel("Discretization (R)")
ax.set_ylabel("Time per experiment (sec)")
ax.set_title(f"ANN-FQ (seismic data) — time distribution across {N_EXP} seeds per R")
ax.set_yscale("log")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("ann_fq_boxplot_time.png", dpi=150)
plt.show()

# --- Box plot: SSE per R ---
sse_data_clean = []
for s in ANN_all_sse:
    arr = np.asarray(s, dtype=float)
    arr = arr[~np.isnan(arr)]
    sse_data_clean.append(arr.tolist())

if any(len(s) > 0 for s in sse_data_clean):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(
        sse_data_clean,
        showmeans=True, meanline=True, patch_artist=True,
        boxprops=dict(facecolor="#185FA5", alpha=0.4, edgecolor="#185FA5"),
        medianprops=dict(color="black", linewidth=1.5),
        meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
        flierprops=dict(marker="o", markersize=3, markerfacecolor="#185FA5", alpha=0.5),
    )
    ax.set_xticklabels([str(int(R)) for R in R_LIST])
    ax.set_xlabel("Discretization (R)")
    ax.set_ylabel("SSE per experiment")
    ax.set_title(f"ANN-FQ (seismic data) — SSE distribution across {N_EXP} seeds per R")
    ax.set_yscale("log")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("ann_fq_boxplot_sse.png", dpi=150)
    plt.show()

print("ANN-FQ box plots saved.")


## RP-ANN-FQ — Random-projection ANN-FQ


In [ ]:
import time, gc, numpy as np

RPANN_all_times = []
RPANN_all_sse   = []
D_RP    = 64
RP_TYPE = "gauss"

print("Collecting RP-ANN-FQ per-experiment times...")
total_start = time.perf_counter()

for iR, R in enumerate(R_LIST, 1):
    print(f"\n========== RP-ANN-FQ | R={R} ({iR}/{len(R_LIST)}) ==========")
    Xr = interpolate_dataset(X, R)
    X2 = np.sum(Xr * Xr, axis=1)
    exp_times, exp_sse = [], []

    for exp in range(N_EXP):
        t0 = time.perf_counter()
        rng = np.random.default_rng(exp)
        d = min(D_RP, R)
        if RP_TYPE == "gauss":
            P = rng.normal(0.0, 1.0, size=(R, d)) / np.sqrt(d)
        else:
            P = rng.choice([-1.0, 1.0], size=(R, d)) / np.sqrt(d)
        idx = rng.choice(Xr.shape[0], size=N_QUANTA, replace=False)
        C = Xr[idx]
        Xp = Xr @ P; Cp = C @ P
        Xp2 = np.sum(Xp * Xp, axis=1, keepdims=True)
        Cp2 = np.sum(Cp * Cp, axis=1, keepdims=True).T
        D2p = Xp2 + Cp2 - 2.0 * (Xp @ Cp.T)
        assign = np.argmin(D2p, axis=1)
        C2 = np.sum(C * C, axis=1)
        XC = np.sum(Xr * C[assign], axis=1)
        sse = float(np.sum(X2 + C2[assign] - 2.0 * XC))
        dt = time.perf_counter() - t0
        exp_times.append(dt); exp_sse.append(sse)
        if (exp + 1) % 20 == 0 or exp == N_EXP - 1:
            print(f"  [R={R}] EXP {exp+1:3d}/{N_EXP} | t={dt:.5f}s | SSE={sse:.3e}")
        del P, C, Xp, Cp, Xp2, Cp2, D2p, assign, C2, XC

    RPANN_all_times.append(exp_times)
    RPANN_all_sse.append(exp_sse)
    m = float(np.mean(exp_times)); s = float(np.std(exp_times, ddof=1))
    print(f"  R={R} DONE | mean={m:.5f}s | std={s:.5f}s | CV={s/m:.4f}")
    del Xr, X2; gc.collect()

print(f"\n========== RP-ANN-FQ COMPLETE ({(time.perf_counter()-total_start)/60:.2f} min) ==========")


In [ ]:
# ============================================================
# BOX PLOTS for RP-ANN-FQ (second application, seismic data)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

assert "RPANN_all_times" in globals(), "RPANN_all_times not found. Run the RP-ANN-FQ block first."

# --- Box plot: time per R ---
time_data = [list(np.asarray(t, dtype=float)) for t in RPANN_all_times]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(
    time_data,
    showmeans=True, meanline=True, patch_artist=True,
    boxprops=dict(facecolor="#993C1D", alpha=0.4, edgecolor="#993C1D"),
    medianprops=dict(color="black", linewidth=1.5),
    meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
    flierprops=dict(marker="o", markersize=3, markerfacecolor="#993C1D", alpha=0.5),
)
ax.set_xticklabels([str(int(R)) for R in R_LIST])
ax.set_xlabel("Discretization (R)")
ax.set_ylabel("Time per experiment (sec)")
ax.set_title(f"RP-ANN-FQ (seismic data) — time distribution across {N_EXP} seeds per R")
ax.set_yscale("log")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("rp_ann_fq_boxplot_time.png", dpi=150)
plt.show()

# --- Box plot: SSE per R ---
sse_data_clean = []
for s in RPANN_all_sse:
    arr = np.asarray(s, dtype=float)
    arr = arr[~np.isnan(arr)]
    sse_data_clean.append(arr.tolist())

if any(len(s) > 0 for s in sse_data_clean):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(
        sse_data_clean,
        showmeans=True, meanline=True, patch_artist=True,
        boxprops=dict(facecolor="#993C1D", alpha=0.4, edgecolor="#993C1D"),
        medianprops=dict(color="black", linewidth=1.5),
        meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
        flierprops=dict(marker="o", markersize=3, markerfacecolor="#993C1D", alpha=0.5),
    )
    ax.set_xticklabels([str(int(R)) for R in R_LIST])
    ax.set_xlabel("Discretization (R)")
    ax.set_ylabel("SSE per experiment")
    ax.set_title(f"RP-ANN-FQ (seismic data) — SSE distribution across {N_EXP} seeds per R")
    ax.set_yscale("log")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("rp_ann_fq_boxplot_sse.png", dpi=150)
    plt.show()

print("RP-ANN-FQ box plots saved.")


## KD-ANN-FQ — KDTree-based ANN-FQ

Note: KD-tree degrades for high-D data; expect it to be slower than RP-ANN-FQ at large R.


In [ ]:
import time, gc, warnings, numpy as np
from sklearn.neighbors import KDTree

KDANN_all_times = []
KDANN_all_sse   = []

print("Collecting KD-ANN-FQ per-experiment times...")
total_start = time.perf_counter()

for iR, R in enumerate(R_LIST, 1):
    print(f"\n========== KD-ANN-FQ | R={R} ({iR}/{len(R_LIST)}) ==========")
    Xr = interpolate_dataset(X, R)
    exp_times, exp_sse = [], []
    for exp in range(N_EXP):
        t0 = time.perf_counter()
        rng = np.random.default_rng(exp)
        sub_idx = rng.choice(Xr.shape[0], size=min(Xr.shape[0], 4 * N_QUANTA), replace=False)
        subset = Xr[sub_idx]
        init_idx = rng.choice(subset.shape[0], size=N_QUANTA, replace=False)
        centers = subset[init_idx]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            tree = KDTree(centers, leaf_size=40, metric="euclidean")
            d, _ = tree.query(Xr, k=1, return_distance=True)
        sse = float(np.sum(d[:, 0] ** 2))
        dt = time.perf_counter() - t0
        exp_times.append(dt); exp_sse.append(sse)
        if (exp + 1) % 20 == 0 or exp == N_EXP - 1:
            print(f"  [R={R}] EXP {exp+1:3d}/{N_EXP} | t={dt:.5f}s | SSE={sse:.3e}")
        del centers, tree, d

    KDANN_all_times.append(exp_times)
    KDANN_all_sse.append(exp_sse)
    m = float(np.mean(exp_times)); s = float(np.std(exp_times, ddof=1))
    print(f"  R={R} DONE | mean={m:.5f}s | std={s:.5f}s | CV={s/m:.4f}")
    del Xr; gc.collect()

print(f"\n========== KD-ANN-FQ COMPLETE ({(time.perf_counter()-total_start)/60:.2f} min) ==========")


In [ ]:
# ============================================================
# BOX PLOTS for KD-ANN-FQ (second application, seismic data)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

assert "KDANN_all_times" in globals(), "KDANN_all_times not found. Run the KD-ANN-FQ block first."

# --- Box plot: time per R ---
time_data = [list(np.asarray(t, dtype=float)) for t in KDANN_all_times]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(
    time_data,
    showmeans=True, meanline=True, patch_artist=True,
    boxprops=dict(facecolor="#0F6E56", alpha=0.4, edgecolor="#0F6E56"),
    medianprops=dict(color="black", linewidth=1.5),
    meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
    flierprops=dict(marker="o", markersize=3, markerfacecolor="#0F6E56", alpha=0.5),
)
ax.set_xticklabels([str(int(R)) for R in R_LIST])
ax.set_xlabel("Discretization (R)")
ax.set_ylabel("Time per experiment (sec)")
ax.set_title(f"KD-ANN-FQ (seismic data) — time distribution across {N_EXP} seeds per R")
ax.set_yscale("log")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("kd_ann_fq_boxplot_time.png", dpi=150)
plt.show()

# --- Box plot: SSE per R ---
sse_data_clean = []
for s in KDANN_all_sse:
    arr = np.asarray(s, dtype=float)
    arr = arr[~np.isnan(arr)]
    sse_data_clean.append(arr.tolist())

if any(len(s) > 0 for s in sse_data_clean):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(
        sse_data_clean,
        showmeans=True, meanline=True, patch_artist=True,
        boxprops=dict(facecolor="#0F6E56", alpha=0.4, edgecolor="#0F6E56"),
        medianprops=dict(color="black", linewidth=1.5),
        meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
        flierprops=dict(marker="o", markersize=3, markerfacecolor="#0F6E56", alpha=0.5),
    )
    ax.set_xticklabels([str(int(R)) for R in R_LIST])
    ax.set_xlabel("Discretization (R)")
    ax.set_ylabel("SSE per experiment")
    ax.set_title(f"KD-ANN-FQ (seismic data) — SSE distribution across {N_EXP} seeds per R")
    ax.set_yscale("log")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("kd_ann_fq_boxplot_sse.png", dpi=150)
    plt.show()

print("KD-ANN-FQ box plots saved.")


## MR-FQ — Multi-resolution functional quantization

Coarse-solve at `R_COARSE`, lift centroids, run a small number of refinement iters at the target R.


In [ ]:
import time, gc, numpy as np
from sklearn.cluster import KMeans

MRFQ_all_times = []
MRFQ_all_sse   = []
R_COARSE         = 512
MAX_ITER_COARSE  = 50
MAX_ITER_FINE    = 5

def lift_centroids(C_old, R_target):
    K, R_old = C_old.shape
    if R_old == R_target:
        return C_old.copy()
    og = np.linspace(0, 1, R_old)
    ng = np.linspace(0, 1, R_target)
    C_new = np.zeros((K, R_target))
    for k in range(K):
        C_new[k] = np.interp(ng, og, C_old[k])
    return C_new

print(f"Collecting MR-FQ per-experiment times (R_COARSE={R_COARSE})...")
X_coarse = interpolate_dataset(X, R_COARSE)
total_start = time.perf_counter()

for iR, R in enumerate(R_LIST, 1):
    print(f"\n========== MR-FQ | R={R} ({iR}/{len(R_LIST)}) ==========")
    Xr = interpolate_dataset(X, R)
    exp_times, exp_sse = [], []
    for exp in range(N_EXP):
        t0 = time.perf_counter()
        kmc = KMeans(n_clusters=N_QUANTA, init="k-means++", n_init=1,
                     max_iter=MAX_ITER_COARSE, tol=TOL, algorithm="lloyd",
                     random_state=exp)
        kmc.fit(X_coarse)
        Cinit = lift_centroids(kmc.cluster_centers_, R)
        kmf = KMeans(n_clusters=N_QUANTA, init=Cinit, n_init=1,
                     max_iter=MAX_ITER_FINE, tol=TOL, algorithm="lloyd",
                     random_state=exp)
        kmf.fit(Xr)
        dt = time.perf_counter() - t0
        exp_times.append(dt); exp_sse.append(float(kmf.inertia_))
        if (exp + 1) % 10 == 0 or exp == N_EXP - 1:
            avg = float(np.mean(exp_times))
            print(f"  [R={R}] EXP {exp+1:3d}/{N_EXP} | t={dt:.4f}s | "
                  f"avg={avg:.4f}s | SSE={kmf.inertia_:.3e}")
        del kmc, Cinit, kmf

    MRFQ_all_times.append(exp_times)
    MRFQ_all_sse.append(exp_sse)
    m = float(np.mean(exp_times)); s = float(np.std(exp_times, ddof=1))
    print(f"  R={R} DONE | mean={m:.4f}s | std={s:.4f}s | CV={s/m:.4f}")
    del Xr; gc.collect()

print(f"\n========== MR-FQ COMPLETE ({(time.perf_counter()-total_start)/60:.2f} min) ==========")


In [ ]:
# ============================================================
# BOX PLOTS for MR-FQ (second application, seismic data)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

assert "MRFQ_all_times" in globals(), "MRFQ_all_times not found. Run the MR-FQ block first."

# --- Box plot: time per R ---
time_data = [list(np.asarray(t, dtype=float)) for t in MRFQ_all_times]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(
    time_data,
    showmeans=True, meanline=True, patch_artist=True,
    boxprops=dict(facecolor="#888780", alpha=0.4, edgecolor="#888780"),
    medianprops=dict(color="black", linewidth=1.5),
    meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
    flierprops=dict(marker="o", markersize=3, markerfacecolor="#888780", alpha=0.5),
)
ax.set_xticklabels([str(int(R)) for R in R_LIST])
ax.set_xlabel("Discretization (R)")
ax.set_ylabel("Time per experiment (sec)")
ax.set_title(f"MR-FQ (seismic data) — time distribution across {N_EXP} seeds per R")
ax.set_yscale("log")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("mr_fq_boxplot_time.png", dpi=150)
plt.show()

# --- Box plot: SSE per R ---
sse_data_clean = []
for s in MRFQ_all_sse:
    arr = np.asarray(s, dtype=float)
    arr = arr[~np.isnan(arr)]
    sse_data_clean.append(arr.tolist())

if any(len(s) > 0 for s in sse_data_clean):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(
        sse_data_clean,
        showmeans=True, meanline=True, patch_artist=True,
        boxprops=dict(facecolor="#888780", alpha=0.4, edgecolor="#888780"),
        medianprops=dict(color="black", linewidth=1.5),
        meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
        flierprops=dict(marker="o", markersize=3, markerfacecolor="#888780", alpha=0.5),
    )
    ax.set_xticklabels([str(int(R)) for R in R_LIST])
    ax.set_xlabel("Discretization (R)")
    ax.set_ylabel("SSE per experiment")
    ax.set_title(f"MR-FQ (seismic data) — SSE distribution across {N_EXP} seeds per R")
    ax.set_yscale("log")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("mr_fq_boxplot_sse.png", dpi=150)
    plt.show()

print("MR-FQ box plots saved.")


## CE-LE — Continuation Elkan (warm-start across R)


In [ ]:
import time, gc, numpy as np
from sklearn.cluster import KMeans

CELE_times = [[] for _ in R_LIST]
CELE_sse   = [[] for _ in R_LIST]

MAX_ITER_FIRST = 100
BURST_ITERS    = 5
MAX_TOTAL_NEXT = 30
EPS_CELE       = 1e-3
PATIENCE       = 2

def lift_centers(C_old, R_target):
    K, R_old = C_old.shape
    if R_old == R_target:
        return C_old.copy()
    og = np.linspace(0, 1, R_old); ng = np.linspace(0, 1, R_target)
    C_new = np.zeros((K, R_target))
    for k in range(K):
        C_new[k] = np.interp(ng, og, C_old[k])
    return C_new

def fit_elkan_bursts(Xr, C_init, max_total, burst, eps, patience, seed):
    total = 0; prev = None; small = 0
    C = C_init; best = None
    while total < max_total:
        n_now = min(burst, max_total - total)
        km = KMeans(n_clusters=C.shape[0], init=C, n_init=1,
                    max_iter=n_now, tol=TOL, algorithm="elkan",
                    random_state=seed)
        km.fit(Xr)
        inertia = float(km.inertia_); C = km.cluster_centers_; total += n_now
        if prev is not None:
            rel = (prev - inertia) / max(prev, 1e-12)
            small = small + 1 if rel < eps else 0
        prev = inertia; best = inertia
        if small >= patience:
            break
    return C, best, total

print("Collecting CE-LE per-experiment times...")
total_start = time.perf_counter()

for exp in range(N_EXP):
    R0 = R_LIST[0]
    Xr0 = interpolate_dataset(X, R0)
    t0 = time.perf_counter()
    km0 = KMeans(n_clusters=N_QUANTA, init="k-means++", n_init=1,
                 max_iter=MAX_ITER_FIRST, tol=TOL, algorithm="elkan",
                 random_state=exp)
    km0.fit(Xr0)
    CELE_times[0].append(time.perf_counter() - t0)
    CELE_sse[0].append(float(km0.inertia_))
    C_prev = km0.cluster_centers_
    del Xr0, km0

    for i in range(1, len(R_LIST)):
        Rn = R_LIST[i]
        Xn = interpolate_dataset(X, Rn)
        Cinit = lift_centers(C_prev, Rn)
        t0 = time.perf_counter()
        Cn, inertia, used = fit_elkan_bursts(
            Xn, Cinit, max_total=MAX_TOTAL_NEXT, burst=BURST_ITERS,
            eps=EPS_CELE, patience=PATIENCE, seed=exp,
        )
        CELE_times[i].append(time.perf_counter() - t0)
        CELE_sse[i].append(inertia)
        C_prev = Cn
        del Xn, Cinit; gc.collect()

    if (exp + 1) % 10 == 0 or exp == N_EXP - 1:
        elapsed = time.perf_counter() - total_start
        avg = elapsed / (exp + 1)
        eta = avg * (N_EXP - exp - 1)
        print(f"  EXP {exp+1:3d}/{N_EXP} | elapsed={elapsed:.1f}s | "
              f"avg={avg:.3f}s/exp | ETA={eta:.1f}s | "
              f"SSE@R_max={CELE_sse[-1][-1]:.3e}")

print(f"\n========== CE-LE COMPLETE ({(time.perf_counter()-total_start)/60:.2f} min) ==========")
for i, R in enumerate(R_LIST):
    m = float(np.mean(CELE_times[i])); s = float(np.std(CELE_times[i], ddof=1))
    print(f"  R={R:>5} | mean={m:.4f}s | std={s:.4f}s | CV={s/m:.4f}")


In [ ]:
# ============================================================
# BOX PLOTS for CE-LE (second application, seismic data)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

assert "CELE_times" in globals(), "CELE_times not found. Run the CE-LE block first."

# --- Box plot: time per R ---
time_data = [list(np.asarray(t, dtype=float)) for t in CELE_times]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(
    time_data,
    showmeans=True, meanline=True, patch_artist=True,
    boxprops=dict(facecolor="#534AB7", alpha=0.4, edgecolor="#534AB7"),
    medianprops=dict(color="black", linewidth=1.5),
    meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
    flierprops=dict(marker="o", markersize=3, markerfacecolor="#534AB7", alpha=0.5),
)
ax.set_xticklabels([str(int(R)) for R in R_LIST])
ax.set_xlabel("Discretization (R)")
ax.set_ylabel("Time per experiment (sec)")
ax.set_title(f"CE-LE (seismic data) — time distribution across {N_EXP} seeds per R")
ax.set_yscale("log")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("ce_le_boxplot_time.png", dpi=150)
plt.show()

# --- Box plot: SSE per R ---
sse_data_clean = []
for s in CELE_sse:
    arr = np.asarray(s, dtype=float)
    arr = arr[~np.isnan(arr)]
    sse_data_clean.append(arr.tolist())

if any(len(s) > 0 for s in sse_data_clean):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(
        sse_data_clean,
        showmeans=True, meanline=True, patch_artist=True,
        boxprops=dict(facecolor="#534AB7", alpha=0.4, edgecolor="#534AB7"),
        medianprops=dict(color="black", linewidth=1.5),
        meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
        flierprops=dict(marker="o", markersize=3, markerfacecolor="#534AB7", alpha=0.5),
    )
    ax.set_xticklabels([str(int(R)) for R in R_LIST])
    ax.set_xlabel("Discretization (R)")
    ax.set_ylabel("SSE per experiment")
    ax.set_title(f"CE-LE (seismic data) — SSE distribution across {N_EXP} seeds per R")
    ax.set_yscale("log")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("ce_le_boxplot_sse.png", dpi=150)
    plt.show()

print("CE-LE box plots saved.")


## CE-MB — Continuation MiniBatch (warm-start across R)


In [ ]:
import time, gc, numpy as np
from sklearn.cluster import MiniBatchKMeans

CE_MB_times = [[] for _ in R_LIST]
CE_MB_sse   = [[] for _ in R_LIST]
MAX_ITER_FIRST_MB = 200
MAX_ITER_NEXT_MB  = 20
BATCH_SIZE_CE     = 1024

def lift_centers_mb(C_old, R_target):
    K, R_old = C_old.shape
    if R_old == R_target:
        return C_old.copy()
    og = np.linspace(0, 1, R_old); ng = np.linspace(0, 1, R_target)
    C_new = np.zeros((K, R_target))
    for k in range(K):
        C_new[k] = np.interp(ng, og, C_old[k])
    return C_new

print("Collecting CE-MB per-experiment times...")
total_start = time.perf_counter()

for exp in range(N_EXP):
    R0 = R_LIST[0]
    Xr0 = interpolate_dataset(X, R0)
    t0 = time.perf_counter()
    mb0 = MiniBatchKMeans(n_clusters=N_QUANTA, init="k-means++", n_init=1,
                          max_iter=MAX_ITER_FIRST_MB, tol=TOL,
                          batch_size=BATCH_SIZE_CE, random_state=exp)
    mb0.fit(Xr0)
    CE_MB_times[0].append(time.perf_counter() - t0)
    CE_MB_sse[0].append(float(mb0.inertia_))
    C_prev = mb0.cluster_centers_
    del Xr0, mb0

    for i in range(1, len(R_LIST)):
        Rn = R_LIST[i]
        Xn = interpolate_dataset(X, Rn)
        Cinit = lift_centers_mb(C_prev, Rn)
        t0 = time.perf_counter()
        mb = MiniBatchKMeans(n_clusters=N_QUANTA, init=Cinit, n_init=1,
                             max_iter=MAX_ITER_NEXT_MB, tol=TOL,
                             batch_size=BATCH_SIZE_CE, random_state=exp)
        mb.fit(Xn)
        CE_MB_times[i].append(time.perf_counter() - t0)
        CE_MB_sse[i].append(float(mb.inertia_))
        C_prev = mb.cluster_centers_
        del Xn, Cinit, mb; gc.collect()

    if (exp + 1) % 10 == 0 or exp == N_EXP - 1:
        elapsed = time.perf_counter() - total_start
        print(f"  EXP {exp+1:3d}/{N_EXP} | elapsed={elapsed:.1f}s | "
              f"SSE@R_max={CE_MB_sse[-1][-1]:.3e}")

print(f"\n========== CE-MB COMPLETE ({(time.perf_counter()-total_start)/60:.2f} min) ==========")
for i, R in enumerate(R_LIST):
    m = float(np.mean(CE_MB_times[i])); s = float(np.std(CE_MB_times[i], ddof=1))
    print(f"  R={R:>5} | mean={m:.4f}s | std={s:.4f}s | CV={s/m:.4f}")


In [ ]:
# ============================================================
# BOX PLOTS for CE-MB (second application, seismic data)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

assert "CE_MB_times" in globals(), "CE_MB_times not found. Run the CE-MB block first."

# --- Box plot: time per R ---
time_data = [list(np.asarray(t, dtype=float)) for t in CE_MB_times]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(
    time_data,
    showmeans=True, meanline=True, patch_artist=True,
    boxprops=dict(facecolor="#993556", alpha=0.4, edgecolor="#993556"),
    medianprops=dict(color="black", linewidth=1.5),
    meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
    flierprops=dict(marker="o", markersize=3, markerfacecolor="#993556", alpha=0.5),
)
ax.set_xticklabels([str(int(R)) for R in R_LIST])
ax.set_xlabel("Discretization (R)")
ax.set_ylabel("Time per experiment (sec)")
ax.set_title(f"CE-MB (seismic data) — time distribution across {N_EXP} seeds per R")
ax.set_yscale("log")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("ce_mb_boxplot_time.png", dpi=150)
plt.show()

# --- Box plot: SSE per R ---
sse_data_clean = []
for s in CE_MB_sse:
    arr = np.asarray(s, dtype=float)
    arr = arr[~np.isnan(arr)]
    sse_data_clean.append(arr.tolist())

if any(len(s) > 0 for s in sse_data_clean):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(
        sse_data_clean,
        showmeans=True, meanline=True, patch_artist=True,
        boxprops=dict(facecolor="#993556", alpha=0.4, edgecolor="#993556"),
        medianprops=dict(color="black", linewidth=1.5),
        meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
        flierprops=dict(marker="o", markersize=3, markerfacecolor="#993556", alpha=0.5),
    )
    ax.set_xticklabels([str(int(R)) for R in R_LIST])
    ax.set_xlabel("Discretization (R)")
    ax.set_ylabel("SSE per experiment")
    ax.set_title(f"CE-MB (seismic data) — SSE distribution across {N_EXP} seeds per R")
    ax.set_yscale("log")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("ce_mb_boxplot_sse.png", dpi=150)
    plt.show()

print("CE-MB box plots saved.")


## SVD-CE-LE — TruncatedSVD low-D proxy + tiny refinement


In [ ]:
import time, gc, numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans

SVDLE_all_times = []
SVDLE_all_sse   = []
P_LATENT     = 32
SVD_ITERS    = 5
REFINE_ITERS = 2

print("Collecting SVD-CE-LE per-experiment times...")
total_start = time.perf_counter()

for iR, R in enumerate(R_LIST, 1):
    print(f"\n========== SVD-CE-LE | R={R} ({iR}/{len(R_LIST)}) ==========")
    Xr = interpolate_dataset(X, R)
    exp_times, exp_sse = [], []
    for exp in range(N_EXP):
        t0 = time.perf_counter()
        svd = TruncatedSVD(n_components=min(P_LATENT, R - 1),
                           n_iter=SVD_ITERS, random_state=exp)
        Z = svd.fit_transform(Xr)
        kml = KMeans(n_clusters=N_QUANTA, init="k-means++", n_init=1,
                     max_iter=50, tol=TOL, algorithm="elkan",
                     random_state=exp)
        labels = kml.fit_predict(Z)
        centers = np.zeros((N_QUANTA, R))
        rng = np.random.default_rng(exp)
        for k in range(N_QUANTA):
            idx = np.where(labels == k)[0]
            centers[k] = Xr[idx].mean(axis=0) if idx.size else Xr[rng.integers(0, Xr.shape[0])]
        kmf = KMeans(n_clusters=N_QUANTA, init=centers, n_init=1,
                     max_iter=REFINE_ITERS, tol=TOL, algorithm="elkan",
                     random_state=exp)
        kmf.fit(Xr)
        dt = time.perf_counter() - t0
        exp_times.append(dt); exp_sse.append(float(kmf.inertia_))
        if (exp + 1) % 10 == 0 or exp == N_EXP - 1:
            avg = float(np.mean(exp_times))
            print(f"  [R={R}] EXP {exp+1:3d}/{N_EXP} | t={dt:.4f}s | "
                  f"avg={avg:.4f}s | SSE={kmf.inertia_:.3e}")
        del svd, Z, kml, labels, centers, kmf

    SVDLE_all_times.append(exp_times)
    SVDLE_all_sse.append(exp_sse)
    m = float(np.mean(exp_times)); s = float(np.std(exp_times, ddof=1))
    print(f"  R={R} DONE | mean={m:.4f}s | std={s:.4f}s | CV={s/m:.4f}")
    del Xr; gc.collect()

print(f"\n========== SVD-CE-LE COMPLETE ({(time.perf_counter()-total_start)/60:.2f} min) ==========")


In [ ]:
# ============================================================
# BOX PLOTS for SVD-CE-LE (second application, seismic data)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

assert "SVDLE_all_times" in globals(), "SVDLE_all_times not found. Run the SVD-CE-LE block first."

# --- Box plot: time per R ---
time_data = [list(np.asarray(t, dtype=float)) for t in SVDLE_all_times]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(
    time_data,
    showmeans=True, meanline=True, patch_artist=True,
    boxprops=dict(facecolor="#7F77DD", alpha=0.4, edgecolor="#7F77DD"),
    medianprops=dict(color="black", linewidth=1.5),
    meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
    flierprops=dict(marker="o", markersize=3, markerfacecolor="#7F77DD", alpha=0.5),
)
ax.set_xticklabels([str(int(R)) for R in R_LIST])
ax.set_xlabel("Discretization (R)")
ax.set_ylabel("Time per experiment (sec)")
ax.set_title(f"SVD-CE-LE (seismic data) — time distribution across {N_EXP} seeds per R")
ax.set_yscale("log")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("svd_ce_le_boxplot_time.png", dpi=150)
plt.show()

# --- Box plot: SSE per R ---
sse_data_clean = []
for s in SVDLE_all_sse:
    arr = np.asarray(s, dtype=float)
    arr = arr[~np.isnan(arr)]
    sse_data_clean.append(arr.tolist())

if any(len(s) > 0 for s in sse_data_clean):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(
        sse_data_clean,
        showmeans=True, meanline=True, patch_artist=True,
        boxprops=dict(facecolor="#7F77DD", alpha=0.4, edgecolor="#7F77DD"),
        medianprops=dict(color="black", linewidth=1.5),
        meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
        flierprops=dict(marker="o", markersize=3, markerfacecolor="#7F77DD", alpha=0.5),
    )
    ax.set_xticklabels([str(int(R)) for R in R_LIST])
    ax.set_xlabel("Discretization (R)")
    ax.set_ylabel("SSE per experiment")
    ax.set_title(f"SVD-CE-LE (seismic data) — SSE distribution across {N_EXP} seeds per R")
    ax.set_yscale("log")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("svd_ce_le_boxplot_sse.png", dpi=150)
    plt.show()

print("SVD-CE-LE box plots saved.")


## RP-CE-LE — Random projection + continuation Elkan

Sena's hero method (Finding 4): expected to be nearly resolution-independent.


In [ ]:
import time, gc, numpy as np
from sklearn.cluster import KMeans

RPCELE_all_times = []
RPCELE_all_sse   = []
D_RP_CE          = 64
RP_TYPE_CE       = "gauss"
MAX_ITER_R0_CE   = 100
MAX_ITER_NEXT_CE = 20

def make_rp(R, d, rng):
    if RP_TYPE_CE == "gauss":
        return rng.normal(0.0, 1.0, size=(R, d)) / np.sqrt(d)
    return rng.choice([-1.0, 1.0], size=(R, d)) / np.sqrt(d)

def lift_C(C_old, R_target):
    K, R_old = C_old.shape
    if R_old == R_target:
        return C_old.copy()
    og = np.linspace(0, 1, R_old); ng = np.linspace(0, 1, R_target)
    C_new = np.zeros((K, R_target))
    for k in range(K):
        C_new[k] = np.interp(ng, og, C_old[k])
    return C_new

print("Collecting RP-CE-LE per-experiment times...")
total_start = time.perf_counter()

# State carried across R for warm-start
prev_centers = [None] * N_EXP

for iR, R in enumerate(R_LIST, 1):
    print(f"\n========== RP-CE-LE | R={R} ({iR}/{len(R_LIST)}) ==========")
    Xr = interpolate_dataset(X, R)
    X2 = np.sum(Xr * Xr, axis=1)
    exp_times, exp_sse = [], []

    for exp in range(N_EXP):
        t0 = time.perf_counter()
        rng = np.random.default_rng(exp * 1000 + R)
        d = min(D_RP_CE, R)
        P = make_rp(R, d, rng)
        Xp = Xr @ P

        if prev_centers[exp] is None:
            init = "k-means++"
            max_iter = MAX_ITER_R0_CE
        else:
            C_prev_orig = prev_centers[exp]
            C_lifted = lift_C(C_prev_orig, R)
            init = C_lifted @ P
            max_iter = MAX_ITER_NEXT_CE

        km = KMeans(n_clusters=N_QUANTA, init=init, n_init=1,
                    max_iter=max_iter, tol=TOL, algorithm="elkan",
                    random_state=exp)
        km.fit(Xp)
        labels = km.labels_

        # Recover original-space centers from labels, compute SSE in original space
        C_orig = np.zeros((N_QUANTA, R))
        rng2 = np.random.default_rng(exp)
        for k in range(N_QUANTA):
            idx = np.where(labels == k)[0]
            C_orig[k] = Xr[idx].mean(axis=0) if idx.size else Xr[rng2.integers(0, Xr.shape[0])]

        C2 = np.sum(C_orig * C_orig, axis=1)
        XC = np.sum(Xr * C_orig[labels], axis=1)
        sse = float(np.sum(X2 + C2[labels] - 2.0 * XC))

        prev_centers[exp] = C_orig

        dt = time.perf_counter() - t0
        exp_times.append(dt); exp_sse.append(sse)
        if (exp + 1) % 20 == 0 or exp == N_EXP - 1:
            print(f"  [R={R}] EXP {exp+1:3d}/{N_EXP} | t={dt:.4f}s | "
                  f"SSE={sse:.3e} | iter={km.n_iter_}")
        del P, Xp, km, labels, C_orig, C2, XC

    RPCELE_all_times.append(exp_times)
    RPCELE_all_sse.append(exp_sse)
    m = float(np.mean(exp_times)); s = float(np.std(exp_times, ddof=1))
    print(f"  R={R} DONE | mean={m:.4f}s | std={s:.4f}s | CV={s/m:.4f}")
    del Xr, X2; gc.collect()

print(f"\n========== RP-CE-LE COMPLETE ({(time.perf_counter()-total_start)/60:.2f} min) ==========")


In [ ]:
# ============================================================
# BOX PLOTS for RP-CE-LE (second application, seismic data)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

assert "RPCELE_all_times" in globals(), "RPCELE_all_times not found. Run the RP-CE-LE block first."

# --- Box plot: time per R ---
time_data = [list(np.asarray(t, dtype=float)) for t in RPCELE_all_times]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(
    time_data,
    showmeans=True, meanline=True, patch_artist=True,
    boxprops=dict(facecolor="#534AB7", alpha=0.4, edgecolor="#534AB7"),
    medianprops=dict(color="black", linewidth=1.5),
    meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
    flierprops=dict(marker="o", markersize=3, markerfacecolor="#534AB7", alpha=0.5),
)
ax.set_xticklabels([str(int(R)) for R in R_LIST])
ax.set_xlabel("Discretization (R)")
ax.set_ylabel("Time per experiment (sec)")
ax.set_title(f"RP-CE-LE (seismic data) — time distribution across {N_EXP} seeds per R")
ax.set_yscale("log")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("rp_ce_le_boxplot_time.png", dpi=150)
plt.show()

# --- Box plot: SSE per R ---
sse_data_clean = []
for s in RPCELE_all_sse:
    arr = np.asarray(s, dtype=float)
    arr = arr[~np.isnan(arr)]
    sse_data_clean.append(arr.tolist())

if any(len(s) > 0 for s in sse_data_clean):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(
        sse_data_clean,
        showmeans=True, meanline=True, patch_artist=True,
        boxprops=dict(facecolor="#534AB7", alpha=0.4, edgecolor="#534AB7"),
        medianprops=dict(color="black", linewidth=1.5),
        meanprops=dict(color="red", linewidth=1.2, linestyle="--"),
        flierprops=dict(marker="o", markersize=3, markerfacecolor="#534AB7", alpha=0.5),
    )
    ax.set_xticklabels([str(int(R)) for R in R_LIST])
    ax.set_xlabel("Discretization (R)")
    ax.set_ylabel("SSE per experiment")
    ax.set_title(f"RP-CE-LE (seismic data) — SSE distribution across {N_EXP} seeds per R")
    ax.set_yscale("log")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("rp_ce_le_boxplot_sse.png", dpi=150)
    plt.show()

print("RP-CE-LE box plots saved.")


## Combined comparison — all methods

Side-by-side box plots across all 15 methods. Skips any method whose arrays are missing (so partial runs still produce a useful figure).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

_REG = [
    ("LX",         "LX_all_times",     "LX_all_sse",    "#378ADD"),
    ("LE",         "LE_all_times",     "LE_all_sse",    "#1D9E75"),
    ("LK",         "LK_all_times",     "LK_all_sse",    "#BA7517"),
    ("KN",         "KN_all_times",     "KN_all_sse",    "#D85A30"),
    ("HT",         "HT_all_times",     "HT_all_sse",    "#7F77DD"),
    ("HK",         "HK_all_times",     "HK_all_sse",    "#D4537E"),
    ("MB",         "MB_all_times",     "MB_all_sse",    "#639922"),
    ("ANN-FQ",     "ANN_all_times",    "ANN_all_sse",   "#185FA5"),
    ("RP-ANN-FQ",  "RPANN_all_times",  "RPANN_all_sse", "#993C1D"),
    ("KD-ANN-FQ",  "KDANN_all_times",  "KDANN_all_sse", "#0F6E56"),
    ("MR-FQ",      "MRFQ_all_times",   "MRFQ_all_sse",  "#888780"),
    ("CE-LE",      "CELE_times",       "CELE_sse",      "#534AB7"),
    ("CE-MB",      "CE_MB_times",      "CE_MB_sse",     "#993556"),
    ("SVD-CE-LE",  "SVDLE_all_times",  "SVDLE_all_sse", "#7F77DD"),
    ("RP-CE-LE",   "RPCELE_all_times", "RPCELE_all_sse","#185FA5"),
]

available = []
for label, tvar, svar, color in _REG:
    if tvar in globals():
        available.append((label, globals()[tvar], globals().get(svar), color))
    else:
        print(f"  [skip] {label} — {tvar} not in session")

if not available:
    raise RuntimeError("No methods found. Run at least one algorithm cell first.")

print(f"\nPlotting {len(available)} methods: {[a[0] for a in available]}\n")

n_methods = len(available); n_R = len(R_LIST)
group_width = 0.8; box_width = group_width / n_methods
positions_base = np.arange(n_R)

# ----- TIME -----
fig, ax = plt.subplots(figsize=(max(11, 1.4 * n_R), 6))
legend_handles = []
for mi, (label, tdata, _, color) in enumerate(available):
    pos = positions_base + (mi - (n_methods - 1) / 2.0) * box_width
    boxes = [list(np.asarray(t, dtype=float)) for t in tdata]
    ax.boxplot(
        boxes, positions=pos, widths=box_width * 0.9,
        patch_artist=True,
        boxprops=dict(facecolor=color, alpha=0.55, edgecolor=color, linewidth=0.8),
        medianprops=dict(color="black", linewidth=1.0),
        whiskerprops=dict(color=color, linewidth=0.8),
        capprops=dict(color=color, linewidth=0.8),
        flierprops=dict(marker="o", markersize=2.5, markerfacecolor=color,
                        markeredgecolor=color, alpha=0.5),
    )
    legend_handles.append(plt.Rectangle((0, 0), 1, 1, facecolor=color,
                                        alpha=0.55, edgecolor=color, label=label))

ax.set_xticks(positions_base)
ax.set_xticklabels([str(int(R)) for R in R_LIST])
ax.set_xlabel("Discretization (R)")
ax.set_ylabel("Time per experiment (sec)")
ax.set_yscale("log")
ax.set_title(f"Seismic-data benchmark — time distribution per R (N_EXP={N_EXP})")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(handles=legend_handles, ncol=min(5, n_methods),
          fontsize=8, loc="upper left", framealpha=0.9, edgecolor="none")
plt.tight_layout()
plt.savefig("secapp_all_methods_boxplot_time.png", dpi=150)
plt.show()

# ----- SSE -----
sse_methods = [(l, t, s, c) for (l, t, s, c) in available if s is not None]
if sse_methods:
    n_methods_s = len(sse_methods); box_width_s = group_width / n_methods_s
    fig, ax = plt.subplots(figsize=(max(11, 1.4 * n_R), 6))
    legend_handles = []
    for mi, (label, _, sdata, color) in enumerate(sse_methods):
        pos = positions_base + (mi - (n_methods_s - 1) / 2.0) * box_width_s
        boxes = []
        for s in sdata:
            arr = np.asarray(s, dtype=float)
            arr = arr[~np.isnan(arr)]
            boxes.append(list(arr) if arr.size else [np.nan])
        ax.boxplot(
            boxes, positions=pos, widths=box_width_s * 0.9,
            patch_artist=True,
            boxprops=dict(facecolor=color, alpha=0.55, edgecolor=color, linewidth=0.8),
            medianprops=dict(color="black", linewidth=1.0),
            whiskerprops=dict(color=color, linewidth=0.8),
            capprops=dict(color=color, linewidth=0.8),
            flierprops=dict(marker="o", markersize=2.5, markerfacecolor=color,
                            markeredgecolor=color, alpha=0.5),
        )
        legend_handles.append(plt.Rectangle((0, 0), 1, 1, facecolor=color,
                                            alpha=0.55, edgecolor=color, label=label))

    ax.set_xticks(positions_base)
    ax.set_xticklabels([str(int(R)) for R in R_LIST])
    ax.set_xlabel("Discretization (R)")
    ax.set_ylabel("SSE per experiment")
    ax.set_yscale("log")
    ax.set_title(f"Seismic-data benchmark — SSE distribution per R (N_EXP={N_EXP})")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(handles=legend_handles, ncol=min(5, n_methods_s),
              fontsize=8, loc="upper left", framealpha=0.9, edgecolor="none")
    plt.tight_layout()
    plt.savefig("secapp_all_methods_boxplot_sse.png", dpi=150)
    plt.show()

print("Saved: secapp_all_methods_boxplot_time.png, secapp_all_methods_boxplot_sse.png")


## Validation tables (Tables 1, 2, 4 of Sena's update — seismic version)

These tables match the structure of Tables 1–4 in `Sena_New_Update.pdf` so
you can put them side-by-side with the lognormal-process numbers and see
where the seismic-data results agree or diverge.

* **Table 1** — Mean wall-clock time per (algorithm, R), bold = fastest at each R.
* **Table 2** — Relative distortion vs LX in **percent** (negative = beats LX).
* **Table 4** — Coefficient of variation of timing across seeds.

CSVs are saved alongside the figures: `secapp_table1_time.csv`,
`secapp_table2_rd.csv`, `secapp_table4_cv.csv`.


In [ ]:
import numpy as np
import pandas as pd

_REG = [
    ("LX",         "LX_all_times",     "LX_all_sse"),
    ("LE",         "LE_all_times",     "LE_all_sse"),
    ("LK",         "LK_all_times",     "LK_all_sse"),
    ("KN",         "KN_all_times",     "KN_all_sse"),
    ("HT",         "HT_all_times",     "HT_all_sse"),
    ("HK",         "HK_all_times",     "HK_all_sse"),
    ("MB",         "MB_all_times",     "MB_all_sse"),
    ("ANN-FQ",     "ANN_all_times",    "ANN_all_sse"),
    ("RP-ANN-FQ",  "RPANN_all_times",  "RPANN_all_sse"),
    ("KD-ANN-FQ",  "KDANN_all_times",  "KDANN_all_sse"),
    ("MR-FQ",      "MRFQ_all_times",   "MRFQ_all_sse"),
    ("CE-LE",      "CELE_times",       "CELE_sse"),
    ("CE-MB",      "CE_MB_times",      "CE_MB_sse"),
    ("SVD-CE-LE",  "SVDLE_all_times",  "SVDLE_all_sse"),
    ("RP-CE-LE",   "RPCELE_all_times", "RPCELE_all_sse"),
]

available = [(lbl, tvar, svar) for lbl, tvar, svar in _REG if tvar in globals()]
print(f"Building tables for {len(available)} methods: {[a[0] for a in available]}")

# ----- Table 1: mean time -----
rows_time = {}
for lbl, tvar, _ in available:
    means = [float(np.mean(t)) for t in globals()[tvar]]
    rows_time[lbl] = means
df_time = pd.DataFrame(rows_time, index=[int(r) for r in R_LIST]).T
df_time.columns = [f"R={r}" for r in df_time.columns]
df_time.index.name = "Method"

# Bold the fastest per R for printing
fastest_per_R = df_time.idxmin(axis=0)

print("\n=== TABLE 1 — Mean wall-clock time (sec), seismic data ===")
print(df_time.round(5).to_string())
print("\nFastest method per R:")
for r in df_time.columns:
    print(f"  {r:>10}: {fastest_per_R[r]}  ({df_time.loc[fastest_per_R[r], r]:.5f} s)")

df_time.to_csv("secapp_table1_time.csv")
print("\nSaved: secapp_table1_time.csv")

# ----- Table 2: relative distortion (%) vs LX -----
if "LX_all_sse" not in globals():
    print("\n[skip] Table 2 — LX SSE not collected, can't compute RD.")
else:
    LX_mean_sse = np.array([np.mean(s) for s in LX_all_sse])
    rows_rd = {}
    for lbl, _, svar in available:
        if svar is None or svar not in globals():
            continue
        means_sse = np.array([np.mean(s) for s in globals()[svar]])
        rd_pct = 100.0 * (means_sse - LX_mean_sse) / LX_mean_sse
        rows_rd[lbl] = rd_pct
    df_rd = pd.DataFrame(rows_rd, index=[int(r) for r in R_LIST]).T
    df_rd.columns = [f"R={r}" for r in df_rd.columns]
    df_rd.index.name = "Method"
    print("\n=== TABLE 2 — Relative distortion (%) vs LX (negative beats LX) ===")
    print(df_rd.round(2).to_string())
    df_rd.to_csv("secapp_table2_rd.csv")
    print("\nSaved: secapp_table2_rd.csv")

# ----- Table 4: CV of time -----
rows_cv = {}
for lbl, tvar, _ in available:
    means = np.array([np.mean(t) for t in globals()[tvar]])
    stds  = np.array([np.std(t, ddof=1) for t in globals()[tvar]])
    cv = np.where(means > 0, stds / means, np.nan)
    rows_cv[lbl] = cv
df_cv = pd.DataFrame(rows_cv, index=[int(r) for r in R_LIST]).T
df_cv.columns = [f"R={r}" for r in df_cv.columns]
df_cv.index.name = "Method"
print("\n=== TABLE 4 — Coefficient of variation of time ===")
print(df_cv.round(4).to_string())
df_cv.to_csv("secapp_table4_cv.csv")
print("\nSaved: secapp_table4_cv.csv")

# ----- Quick validation summary against Sena's findings -----
print("\n" + "=" * 60)
print("QUICK CROSS-CHECK vs Sena's first-app findings")
print("=" * 60)

if "RPCELE_all_times" in globals():
    rp_times = np.array([np.mean(t) for t in RPCELE_all_times])
    ratio = rp_times[-1] / rp_times[0]
    print(f"\nFinding 4 (RP-CE-LE near-resolution-independence):")
    print(f"  RP-CE-LE time at R={R_LIST[0]}: {rp_times[0]:.4f}s")
    print(f"  RP-CE-LE time at R={R_LIST[-1]}: {rp_times[-1]:.4f}s")
    print(f"  Ratio (R_max / R_min): {ratio:.2f}x")
    print(f"  First-app ratio reported by Sena: ~2.1x")
    if "RPCELE_all_sse" in globals() and "LX_all_sse" in globals():
        rp_sse = np.array([np.mean(s) for s in RPCELE_all_sse])
        lx_sse = np.array([np.mean(s) for s in LX_all_sse])
        rd_rp = 100 * (rp_sse - lx_sse) / lx_sse
        print(f"  RD penalty range (seismic): {rd_rp.min():+.2f}% to {rd_rp.max():+.2f}%")
        print(f"  First-app reported by Sena: +4.78% to +4.85%")

if "KN_all_sse" in globals() and "LX_all_sse" in globals():
    kn_sse = np.array([np.mean(s) for s in KN_all_sse])
    lx_sse = np.array([np.mean(s) for s in LX_all_sse])
    rd_kn = 100 * (kn_sse - lx_sse) / lx_sse
    print(f"\nFinding 3 (KN constant ~+61.7% RD on first-app):")
    print(f"  KN RD on seismic data: {rd_kn.mean():+.2f}% +/- {rd_kn.std():.2f}%")
    print(f"  Range: {rd_kn.min():+.2f}% to {rd_kn.max():+.2f}%")

if "LX_all_times" in globals() and "LK_all_times" in globals():
    lx_t = np.array([np.mean(t) for t in LX_all_times])
    lk_t = np.array([np.mean(t) for t in LK_all_times])
    ratio = lk_t / lx_t
    print(f"\nFinding 5 (LK overhead vs LX, expected 1.7x-16x on first-app):")
    print(f"  LK / LX time ratio range (seismic): {ratio.min():.2f}x to {ratio.max():.2f}x")
    for r, rt in zip(R_LIST, ratio):
        print(f"    R={r:>5}: {rt:.2f}x")

if "MB_all_sse" in globals() and "LX_all_sse" in globals():
    mb_sse = np.array([np.mean(s) for s in MB_all_sse])
    lx_sse = np.array([np.mean(s) for s in LX_all_sse])
    rd_mb = 100 * (mb_sse - lx_sse) / lx_sse
    print(f"\nFinding 6 (MB constant ~+0.67% RD on first-app):")
    print(f"  MB RD on seismic data: {rd_mb.mean():+.3f}% +/- {rd_mb.std():.3f}%")
    print(f"  Range: {rd_mb.min():+.3f}% to {rd_mb.max():+.3f}%")


---
## Appendix — original Step 2 / Step 3 cells

These cells visualise the quantizer output (top-weighted scenarios,
weighted-mean Sa map vs full MCS) for a single LX/LE run. Kept here
for reference; not used by the validation tables above.


In [ ]:
# STEP 2 — Lloyd / Elkan clustering on the seismic IM maps
# Uses X_log from Step 1

import numpy as np
import time
from sklearn.cluster import KMeans

def run_hq_quantizer(
    X_log,
    N=50,
    algorithm="lloyd",   # "lloyd" or "elkan"
    max_iter=20,
    tol=1e-4,
    seed=11,
):
    """
    Run the HQ clustering step on log-IM maps.

    Parameters
    ----------
    X_log : ndarray, shape (Nsim, R)
        Flattened log-intensity maps.
    N : int
        Number of quanta.
    algorithm : str
        "lloyd" for LX, "elkan" for LE.
    max_iter : int
        Max iterations.
    tol : float
        Convergence tolerance.
    seed : int
        Random seed for initialization.

    Returns
    -------
    result : dict
        labels, quanta, weights, distortion, runtime, n_iter, etc.
    """
    if algorithm not in {"lloyd", "elkan"}:
        raise ValueError("algorithm must be 'lloyd' or 'elkan'")

    X = np.asarray(X_log, dtype=np.float64)
    nsim, R = X.shape

    # Paper logic starts from random initial quanta for LX.
    # We keep init='random' and n_init=1 so this is closer to that baseline.
    km = KMeans(
        n_clusters=N,
        init="random",
        n_init=1,
        max_iter=max_iter,
        tol=tol,
        algorithm=algorithm,
        random_state=seed,
    )

    t0 = time.perf_counter()
    km.fit(X)
    runtime = time.perf_counter() - t0

    labels = km.labels_.copy()
    quanta = km.cluster_centers_.copy()   # shape (N, R)

    # Distortion Δ = sum of squared distances to assigned centroid
    # sklearn's inertia_ is exactly this quantity
    distortion = float(km.inertia_)

    # Cluster sizes and probability masses p_j = N_j / Nsim
    counts = np.bincount(labels, minlength=N)
    weights = counts / nsim

    # Sort quanta by descending weight for easier interpretation
    order = np.argsort(weights)[::-1]
    inverse_order = np.empty_like(order)
    inverse_order[order] = np.arange(N)

    quanta = quanta[order]
    weights = weights[order]
    counts = counts[order]
    labels = inverse_order[labels]

    result = {
        "algorithm": algorithm,
        "N": N,
        "Nsim": nsim,
        "R": R,
        "labels": labels,               # shape (Nsim,)
        "quanta_log": quanta,           # shape (N, R)
        "weights": weights,             # shape (N,)
        "counts": counts,               # shape (N,)
        "distortion": distortion,
        "runtime_sec": runtime,
        "n_iter": int(km.n_iter_),
        "converged_before_max_iter": int(km.n_iter_) < max_iter,
        "weights_sum": float(weights.sum()),
        "empty_clusters": int(np.sum(counts == 0)),
        "model": km,
    }
    return result


def print_quantizer_summary(res, name="Result"):
    print(f"\n{name}")
    print("-" * len(name))
    print("Algorithm:", res["algorithm"])
    print("N:", res["N"])
    print("Nsim:", res["Nsim"])
    print("R:", res["R"])
    print("Runtime (s):", round(res["runtime_sec"], 4))
    print("Iterations used:", res["n_iter"])
    print("Stopped before max_iter:", res["converged_before_max_iter"])
    print("Distortion:", f"{res['distortion']:.6e}")
    print("Weights sum:", res["weights_sum"])
    print("Empty clusters:", res["empty_clusters"])
    print("Top 10 weights:", np.round(res["weights"][:10], 4))


# ----------------------------
# Run baseline Lloyd and Elkan
# ----------------------------

res_lx_50 = run_hq_quantizer(
    X_log,
    N=50,
    algorithm="lloyd",
    max_iter=20,
    tol=1e-4,
    seed=11,
)

res_le_50 = run_hq_quantizer(
    X_log,
    N=50,
    algorithm="elkan",
    max_iter=20,
    tol=1e-4,
    seed=11,
)

print_quantizer_summary(res_lx_50, name="LX / Lloyd / N=50")
print_quantizer_summary(res_le_50, name="LE / Elkan / N=50")


# ----------------------------
# Quick consistency checks
# ----------------------------

assert res_lx_50["quanta_log"].shape == (50, X_log.shape[1])
assert res_le_50["quanta_log"].shape == (50, X_log.shape[1])
assert res_lx_50["labels"].shape == (X_log.shape[0],)
assert res_le_50["labels"].shape == (X_log.shape[0],)
assert np.isclose(res_lx_50["weights"].sum(), 1.0)
assert np.isclose(res_le_50["weights"].sum(), 1.0)

print("\nAll checks passed.")


In [ ]:
# STEP 3 — reshape quanta into 2D seismic maps and visualize top weighted scenarios

import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------
# Choose which quantizer result you want to inspect
# -------------------------------------------------
res = res_lx_50   # later we can switch to res_le_50 or N=200

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def reshape_flat_map(flat_map, ny=NY, nx=NX):
    return np.asarray(flat_map).reshape(ny, nx)

def weighted_mean_map_from_quanta(quanta_log, weights):
    """
    Weighted mean in Sa-space:
      E[Sa] ≈ sum_j p_j * exp(f_j)
    where f_j is a log-Sa quantum map.
    """
    quanta_sa = np.exp(quanta_log)
    mean_sa_flat = np.sum(weights[:, None] * quanta_sa, axis=0)
    return reshape_flat_map(mean_sa_flat)

def full_mcs_mean_map(X_log):
    """
    Full Monte Carlo mean in Sa-space.
    """
    mean_sa_flat = np.exp(X_log).mean(axis=0)
    return reshape_flat_map(mean_sa_flat)

def plot_single_map(Z, title="", cmap="viridis", colorbar_label="Sa(T=0.1s) [g]"):
    plt.figure(figsize=(6, 5))
    im = plt.pcolormesh(lon_grid, lat_grid, Z, shading="auto", cmap=cmap)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title(title)
    cbar = plt.colorbar(im)
    cbar.set_label(colorbar_label)
    plt.tight_layout()
    plt.show()

def plot_top_quanta_maps(res, n_show=9):
    quanta_log = res["quanta_log"]
    weights = res["weights"]
    n_show = min(n_show, len(weights))

    ncols = 3
    nrows = int(np.ceil(n_show / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.8 * nrows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()

    # convert selected quanta from log-space to Sa-space for plotting
    quanta_sa = np.exp(quanta_log[:n_show])

    vmin = quanta_sa.min()
    vmax = quanta_sa.max()

    for i in range(n_show):
        ax = axes[i]
        Z = reshape_flat_map(quanta_sa[i])

        pcm = ax.pcolormesh(
            lon_grid, lat_grid, Z,
            shading="auto",
            cmap="viridis",
            vmin=vmin,
            vmax=vmax
        )
        ax.set_title(f"Quantum {i+1}\nweight = {weights[i]:.4f}")
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    for j in range(n_show, len(axes)):
        axes[j].axis("off")

    cbar = fig.colorbar(pcm, ax=axes[:n_show], shrink=0.9)
    cbar.set_label("Sa(T=0.1s) [g]")

    fig.suptitle(f"Top {n_show} representative scenario maps ({res['algorithm']}, N={res['N']})", y=1.02)
    plt.show()


# -------------------------------------------------
# 1. Plot top representative maps
# -------------------------------------------------
plot_top_quanta_maps(res, n_show=9)

# -------------------------------------------------
# 2. Compare weighted mean quantizer map vs full MCS
# -------------------------------------------------
mean_sa_quant = weighted_mean_map_from_quanta(res["quanta_log"], res["weights"])
mean_sa_full = full_mcs_mean_map(X_log)
mean_sa_diff = mean_sa_quant - mean_sa_full

plot_single_map(
    mean_sa_quant,
    title=f"Weighted mean Sa map from quantizer ({res['algorithm']}, N={res['N']})"
)

plot_single_map(
    mean_sa_full,
    title="Weighted mean Sa map from full MCS"
)

plot_single_map(
    mean_sa_diff,
    title="Quantizer mean Sa map - Full MCS mean Sa map",
    cmap="coolwarm",
    colorbar_label="Difference in mean Sa [g]"
)

# -------------------------------------------------
# 3. Small numeric sanity checks
# -------------------------------------------------
print("Quantizer mean Sa map: min/max =", float(mean_sa_quant.min()), float(mean_sa_quant.max()))
print("Full MCS mean Sa map:   min/max =", float(mean_sa_full.min()), float(mean_sa_full.max()))
print("Difference map:         min/max =", float(mean_sa_diff.min()), float(mean_sa_diff.max()))
print("Difference map:         mean abs diff =", float(np.mean(np.abs(mean_sa_diff))))
